In [4]:
import os
import shutil

import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

from skimage.filters import frangi
from skimage.morphology import skeletonize


# ============================================================
# 入出力設定
# ============================================================

INPUT_GREEN_DIR = "preprocessed_green_images"

# 前処理済みGreen画像を、そのまま血管抽出の入力として使用する。
# Trueの場合、血管抽出側ではCLAHE・再正規化・追加の照明補正を行わない。
# これにより、fundus_preprocessing.pyの前処理を二重適用することを防ぐ。
USE_PREPROCESSED_DIRECT = True
INPUT_MASK_DIR = "fundus_masks"
ORIGINAL_IMAGE_DIR = "healthy_images"

OUTPUT_DIR = "vessel_extracted_images"
DEBUG_DIR = "vessel_debug_samples"
OUTPUT_CSV = "vessel_extraction_metrics.csv"

TEMP_OUTPUT_DIR = OUTPUT_DIR + "_tmp"
TEMP_DEBUG_DIR = DEBUG_DIR + "_tmp"
TEMP_CSV = OUTPUT_CSV + ".tmp"

DEBUG_SAMPLE_COUNT = 5

# デバッグは常に先頭5枚だけを対象とする。
# ランダム抽出は行わない。
ALWAYS_DEBUG_COUNT = 5


# ============================================================
# 処理解像度
# ============================================================

# 細い血管を消さないため、原則として縮小しない。
PROCESS_SCALE = 1.0


# ============================================================
# Fundus mask
# ============================================================

# 保存済み眼底マスクの境界から少し内側を使用する。
EFFECTIVE_MASK_ERODE_SIZE = 5


# ============================================================
# Frangi
# ============================================================

FRANGI_SIGMAS = (
    0.7,
    1.0,
    1.4,
    2.0,
    2.8,
    4.0,
)

# 0.7pxだけで反応する細かな模様は、血管よりも高周波ノイズになりやすい。
# ただし細い血管を完全に捨てないため、fine-scale responseは少し弱めて残す。
FINE_FRANGI_WEIGHT = 0.82

FRANGI_ALPHA = 0.5
FRANGI_BETA = 0.5
FRANGI_GAMMA = None


# ============================================================
# Green画像の平滑化
# ============================================================

GREEN_NOISE_HIGH_THRESHOLD = 14.0

BLUR_KERNEL_NORMAL = (3, 3)

# 細い血管を消しすぎないため、ノイズ画像でも3x3を使用する。
# ノイズが非常に強いデータセットでは(5, 5)も検討する。
BLUR_KERNEL_NOISY = (3, 3)

# 暗い/ノイズの多い画像では、細かな眼底テクスチャが
# Frangiへ強く反応しやすいため、画像状態に応じて
# 前処理と候補判定を適応させる。
ADAPTIVE_DARK_MEAN = 95.0
ADAPTIVE_DARK_MEDIAN = 92.0
ADAPTIVE_NOISE_STD = 6.0
ADAPTIVE_NOISE_RATIO = 0.075
ADAPTIVE_CONDITION_THRESHOLD = 0.35

# 暗い画像の同心円バンディングを、fundus中心からの半径方向の
# 背景プロファイルとして推定して除去する。
# 血管は円周全体には存在しないため、半径ごとの中央値は
# 血管そのものよりも背景バンディングを捉えやすい。
RADIAL_BANDING_ENABLED = True
RADIAL_BANDING_MIN_CONDITION = 0.25
RADIAL_BANDING_STRENGTH = 0.90
RADIAL_PROFILE_SMOOTH_RADIUS = 15

ADAPTIVE_BILATERAL_D = 5
ADAPTIVE_BILATERAL_SIGMA_COLOR = 18.0
ADAPTIVE_BILATERAL_SIGMA_SPACE = 3.0

CLAHE_CLIP_NORMAL = 1.2
CLAHE_CLIP_DARK = 0.75

COHERENCE_SIGMA_NORMAL = 12.0
COHERENCE_SIGMA_DARK = 16.0


# ============================================================
# Black-hat
# ============================================================

BLACKHAT_KERNEL_SIZES = (
    11,
    17,
    25,
    31,
)


# ============================================================
# 局所暗線
# ============================================================

DARK_BACKGROUND_SIGMA = 9.0


# ============================================================
# Frangi threshold
# ============================================================

# 強い血管のseed。
FRANGI_HIGH_PERCENTILE = 86.0

# 弱い血管候補。
FRANGI_LOW_PERCENTILE = 55.0

BLACKHAT_SUPPORT_PERCENTILE = 70.0
DARK_SUPPORT_PERCENTILE = 58.0


# ============================================================
# Frangiノイズ判定
# ============================================================

FRANGI_NOISE_RATIO_LIMIT = 0.48
FRANGI_PERCENTILE_MAX_BOOST = 4.0


# ============================================================
# 局所コントラスト正規化（細い周辺血管の救済用）
#
# グローバルなpercentileだけでは、
# もともとコントラストが低い周辺部の細い血管を
# 拾おうとすると、必ずノイズも一緒に増える。
#
# 「局所平均に対する比率」で見ることで、
# 画像全体の明るさ・コントラストのムラに関わらず、
# その場所なりに突出した線状構造だけを拾う。
# ============================================================

LOCAL_CONTRAST_RADIUS = 31

# 局所平均の何倍以上を「局所的に突出している」とみなすか。
LOCAL_CONTRAST_RATIO_THRESHOLD = 2.2

# 追加
LOCAL_CONTRAST_MIN_FRANGI = 16

# ============================================================
# Green画像の局所平均を使った薄い血管救済
# ============================================================
# 画像全体の平均ではなく、8x8に分割した各領域の平均輝度を
# 求め、その周辺平均より暗い画素を「局所的な暗線候補」とする。
#
# ただし「平均より暗い」だけでは背景の濃淡や眼底境界も拾うため、
# 最終採用ではcoherence / Frangi / dark / Black-hatと組み合わせる。
LOCAL_DARK_GRID_SIZE = 16
LOCAL_DARK_MIN_DROP = 4.5
LOCAL_DARK_STD_FACTOR = 0.45
LOCAL_DARK_MIN_Z = 0.75
LOCAL_DARK_MIN_FRANGI = 2.5
LOCAL_DARK_MIN_COHERENCE = 0.30
LOCAL_DARK_SUPPORT_PERCENTILE = 38.0

# ============================================================
# 暗い画像で発生する同心円状アーティファクト対策
# ============================================================
# Fundus中心から見て「円周方向」に並ぶ弱い線を検出する。
# 本物の血管も曲がるため、強い血管は削除せず、
# 暗い画像の弱い候補だけにペナルティをかける。
RING_ARTIFACT_ENABLED = True
RING_ARTIFACT_CONDITION_MIN = 0.35
RING_ARTIFACT_SCORE_THRESHOLD = 0.80
RING_ARTIFACT_MIN_RADIUS_RATIO = 0.10
RING_ARTIFACT_MAX_RADIUS_RATIO = 1.05
RING_ARTIFACT_SUPPRESS_WEAK_FACTOR = 1.05

# ============================================================
# 方向一貫性
# ============================================================

# 構造テンソルの平滑化スケール。
COHERENCE_SIGMA = 16.0

# 0～1。
# 高くするとノイズは減るが、曲がった細い血管も減りやすい。
COHERENCE_MIN = 0.34


# ============================================================
# Geodesic hysteresis
# ============================================================

HYSTERESIS_MAX_ITERATIONS = 512
HYSTERESIS_KERNEL_SIZE = 3


# ============================================================
# 候補領域の接続
# ============================================================

CONNECTION_KERNEL_SIZE = 1


# ============================================================
# Strong seedからの薄い血管成長
# ============================================================
# 現在のcandidateは「血管らしさ」を優先しているため、
# 暗い画像では本物の細い血管が途中でcandidateから抜ける。
# そこでstrong seedを起点に、少し弱い線状応答まで段階的に
# 取り込み、血管ネットワークとして連続させる。
SEED_GROWTH_ENABLED = True

# 旧版の「全方向dilation + 160回成長」は、暗い画像の
# 同心円状テクスチャまでseedから伝播させてしまった。
# 新版では、strong seed / 既存血管の「端点」からだけ成長させる。
SEED_GROWTH_MAX_ITERATIONS = 80
SEED_GROWTH_KERNEL_SIZE = 3
SEED_GROWTH_CLOSE_SIZE = 1

# 成長候補の閾値。旧版より少し厳しくし、
# 「弱ければ何でも」ではなく複数の血管特徴が揃うことを要求する。
SEED_GROWTH_FRANGI_PERCENTILE = 45.0
SEED_GROWTH_DARK_PERCENTILE = 48.0
SEED_GROWTH_BLACKHAT_PERCENTILE = 52.0
SEED_GROWTH_COHERENCE_MIN = 0.30
SEED_GROWTH_LOCAL_RATIO_MIN = 1.25

# 端点から一度に遠くへ飛ばない。
SEED_GROWTH_MAX_STEP_DISTANCE = 1

# 端点の過去方向を推定する距離。
SEED_GROWTH_TIP_LOOKBACK = 10

# 端点の進行方向周辺だけを探索する。
SEED_GROWTH_SEARCH_RADIUS = 3

# 進行方向との一致度。1.0=完全一致、0=直交。
SEED_GROWTH_DIRECTION_COSINE = 0.45

# 成長画素に最低限要求する複合スコア。
SEED_GROWTH_MIN_SCORE = 0.92

# 1画素だけの偶然の反応を避けるため、
# 近傍に追加候補が存在する場合を優先する。
SEED_GROWTH_MIN_SUPPORT_NEIGHBORS = 2


# ============================================================
# 小成分除去
# ============================================================

MIN_COMPONENT_AREA = 8
MIN_COMPONENT_LENGTH = 7
MIN_COMPONENT_MEAN_CONFIDENCE = 8.0

# skeleton長 / sqrt(area)。
MIN_COMPONENT_ELONGATION = 1.8

# 塊状成分の判定に使用する。
MAX_COMPACT_FILL_RATIO = 0.35


# ============================================================
# 局所高密度ノイズ除去
# ============================================================

# 正常な血管分岐まで消す場合があるため、初期値では無効。
ENABLE_DENSE_NOISE_SUPPRESSION = False

DENSE_NOISE_WINDOW_SIZE = 41
DENSE_NOISE_MAX_DENSITY = 0.45
DENSE_NOISE_MAX_DENSITY_STRICT = 0.28


# ============================================================
# Mask境界対策
# ============================================================

BORDER_SUPPRESSION_WIDTH = 26


# ============================================================
# 中心線化
# ============================================================

CENTERLINE_CLOSE_SIZE = 3


# ============================================================
# Overlay
# ============================================================

OVERLAY_ALPHA = 0.90

# 最終結果のoverlayは、細いskeletonそのものではなく
# 見やすさのために少し太く描画する。血管マスク自体は変更しない。
OVERLAY_LINE_THICKNESS = 4

# デバッグ画像ではさらに太くして、抽出位置を確認しやすくする。
DEBUG_OVERLAY_LINE_THICKNESS = 6


# ============================================================
# 境界セーフなモルフォロジー
# ============================================================

def safe_erode(mask, kernel, iterations=1):
    """
    画像外を0でパディングしてから収縮する。

    眼底マスクが画像端に接している場合でも、
    画像端を背景として正しく収縮させる。
    """

    kernel_height, kernel_width = kernel.shape[:2]
    pad = max(kernel_height, kernel_width)

    padded = cv2.copyMakeBorder(
        mask,
        pad,
        pad,
        pad,
        pad,
        borderType=cv2.BORDER_CONSTANT,
        value=0,
    )

    eroded = cv2.erode(
        padded,
        kernel,
        iterations=iterations,
    )

    return eroded[
        pad:pad + mask.shape[0],
        pad:pad + mask.shape[1],
    ]


def safe_close(mask, kernel):
    """
    画像外を0でパディングしてからモルフォロジーCloseを行う。
    """

    kernel_height, kernel_width = kernel.shape[:2]
    pad = max(kernel_height, kernel_width)

    padded = cv2.copyMakeBorder(
        mask,
        pad,
        pad,
        pad,
        pad,
        borderType=cv2.BORDER_CONSTANT,
        value=0,
    )

    closed = cv2.morphologyEx(
        padded,
        cv2.MORPH_CLOSE,
        kernel,
    )

    return closed[
        pad:pad + mask.shape[0],
        pad:pad + mask.shape[1],
    ]


# ============================================================
# 共通関数
# ============================================================

def clear_directory(directory):
    """
    フォルダ自体は残し、中身をすべて削除する。
    """

    os.makedirs(
        directory,
        exist_ok=True,
    )

    for name in os.listdir(directory):
        path = os.path.join(
            directory,
            name,
        )

        try:
            if (
                os.path.isdir(path)
                and not os.path.islink(path)
            ):
                shutil.rmtree(path)
            else:
                os.remove(path)

        except Exception as exc:
            print(
                f"削除できませんでした: "
                f"{path} ({exc})"
            )


def mask_image(image, mask):
    """
    mask外を0にする。

    2次元画像と3次元画像の両方に対応する。
    """

    if image.ndim == 2:
        return cv2.bitwise_and(
            image,
            image,
            mask=mask,
        )

    result = image.copy()
    result[mask == 0] = 0

    return result


def save_image(path, image):
    """
    cv2.imwriteの成功を確認する。
    """

    directory = os.path.dirname(path)

    if directory:
        os.makedirs(
            directory,
            exist_ok=True,
        )

    success = cv2.imwrite(
        path,
        image,
    )

    if not success:
        raise IOError(
            f"画像を保存できませんでした: {path}"
        )


def get_base_image_id(filename):
    """
    ファイル名から画像IDを取り出す。

    例:
        abc_green_preprocessed.png -> abc
        abc_green.png              -> abc
        abc_preprocessed.png       -> abc
        abc.png                    -> abc
    """

    stem = os.path.splitext(
        filename
    )[0]

    suffixes = (
        "_green_preprocessed",
        "_green",
        "_preprocessed",
    )

    for suffix in suffixes:
        if stem.endswith(suffix):
            stem = stem[:-len(suffix)]
            break

    return stem


# ============================================================
# 元画像検索
# ============================================================

def find_original_image(base_id):
    extensions = (
        ".png",
        ".jpg",
        ".jpeg",
        ".bmp",
        ".tif",
        ".tiff",
        ".PNG",
        ".JPG",
        ".JPEG",
        ".BMP",
        ".TIF",
        ".TIFF",
    )

    for extension in extensions:
        path = os.path.join(
            ORIGINAL_IMAGE_DIR,
            base_id + extension,
        )

        if os.path.isfile(path):
            return path

    return None


# ============================================================
# Fundus mask検索
# ============================================================

def find_saved_fundus_mask(base_id):
    candidates = (
        base_id + "_fundus_mask.png",
        base_id + "_mask.png",
        base_id + ".png",
    )

    for name in candidates:
        path = os.path.join(
            INPUT_MASK_DIR,
            name,
        )

        if os.path.isfile(path):
            return path

    if os.path.isdir(INPUT_MASK_DIR):
        target_stems = {
            os.path.splitext(name)[0]
            for name in candidates
        }

        for name in os.listdir(INPUT_MASK_DIR):
            stem, extension = os.path.splitext(name)

            if (
                extension.lower() == ".png"
                and stem in target_stems
            ):
                path = os.path.join(
                    INPUT_MASK_DIR,
                    name,
                )

                if os.path.isfile(path):
                    return path

    return None


# ============================================================
# Fundus mask最大成分
# ============================================================

def keep_largest_mask_component(mask):
    """
    保存済みmaskの最大連結成分だけを使用する。
    """

    binary = np.where(
        mask > 0,
        255,
        0,
    ).astype(np.uint8)

    num_labels, labels, stats, _ = (
        cv2.connectedComponentsWithStats(
            binary,
            connectivity=8,
        )
    )

    if num_labels <= 1:
        return binary

    areas = stats[
        1:,
        cv2.CC_STAT_AREA,
    ]

    largest_label = 1 + int(
        np.argmax(areas)
    )

    largest_area = int(
        stats[
            largest_label,
            cv2.CC_STAT_AREA,
        ]
    )

    image_area = (
        binary.shape[0]
        * binary.shape[1]
    )

    if largest_area < image_area * 0.01:
        return binary

    output = np.where(
        labels == largest_label,
        255,
        0,
    ).astype(np.uint8)

    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (7, 7),
    )

    output = safe_close(
        output,
        kernel,
    )

    return np.where(
        output > 0,
        255,
        0,
    ).astype(np.uint8)


# ============================================================
# Fundus mask読み込み
# ============================================================

def load_saved_fundus_mask(
    base_id,
    target_shape,
):
    mask_path = find_saved_fundus_mask(
        base_id
    )

    if mask_path is None:
        return None, None

    mask = cv2.imread(
        mask_path,
        cv2.IMREAD_GRAYSCALE,
    )

    if mask is None:
        return None, mask_path

    if mask.shape != target_shape:
        mask = cv2.resize(
            mask,
            (
                target_shape[1],
                target_shape[0],
            ),
            interpolation=cv2.INTER_NEAREST,
        )

    mask = np.where(
        mask > 0,
        255,
        0,
    ).astype(np.uint8)

    mask = keep_largest_mask_component(
        mask
    )

    if np.count_nonzero(mask) == 0:
        return None, mask_path

    return mask, mask_path


# ============================================================
# 画像端でマスクが切れているか確認
# ============================================================

def mask_touches_border(mask):
    if mask.size == 0:
        return False

    touches = (
        np.any(mask[0, :] > 0)
        or np.any(mask[-1, :] > 0)
        or np.any(mask[:, 0] > 0)
        or np.any(mask[:, -1] > 0)
    )

    return bool(touches)


# ============================================================
# Effective mask
# ============================================================

def create_effective_mask(fundus_mask):
    if EFFECTIVE_MASK_ERODE_SIZE <= 1:
        return fundus_mask.copy()

    kernel_size = EFFECTIVE_MASK_ERODE_SIZE

    if kernel_size % 2 == 0:
        kernel_size += 1

    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (
            kernel_size,
            kernel_size,
        ),
    )

    effective_mask = safe_erode(
        fundus_mask,
        kernel,
        iterations=1,
    )

    effective_mask = np.where(
        effective_mask > 0,
        255,
        0,
    ).astype(np.uint8)

    # 収縮によってマスクが消えた場合は元マスクを使用する。
    if np.count_nonzero(effective_mask) == 0:
        return fundus_mask.copy()

    return effective_mask


# ============================================================
# Mask外を中央値で埋める
# ============================================================

def fill_outside_with_median(
    image,
    mask,
):
    result = image.astype(
        np.float32
    ).copy()

    valid = mask > 0

    if not np.any(valid):
        return result

    median_value = float(
        np.median(
            result[valid]
        )
    )

    result[~valid] = median_value

    return result


# ============================================================
# Green channel正規化
# ============================================================

def normalize_inside_mask(
    image,
    mask,
):
    output = np.zeros_like(
        image,
        dtype=np.uint8,
    )

    pixels = image[
        mask > 0
    ]

    if pixels.size == 0:
        return output

    percentile_1 = float(
        np.percentile(
            pixels,
            1.0,
        )
    )

    percentile_99 = float(
        np.percentile(
            pixels,
            99.0,
        )
    )

    if percentile_99 <= percentile_1 + 1e-6:
        return output

    normalized = (
        image.astype(np.float32)
        - percentile_1
    ) / (
        percentile_99
        - percentile_1
    )

    normalized = np.clip(
        normalized,
        0.0,
        1.0,
    )

    output = (
        normalized * 255.0
    ).astype(np.uint8)

    return mask_image(
        output,
        mask,
    )


# ============================================================
# Green画像のノイズレベル
# ============================================================

def estimate_green_noise(
    green,
    mask,
):
    """
    Laplacian応答の標準偏差から、
    Green画像の細かなざらつきを推定する。
    """

    if np.count_nonzero(mask) == 0:
        return 0.0

    laplacian = cv2.Laplacian(
        green,
        cv2.CV_64F,
        ksize=3,
    )

    values = laplacian[
        mask > 0
    ]

    if values.size == 0:
        return 0.0

    return float(
        np.std(values)
    )


def estimate_image_condition(
    green,
    mask,
):
    """暗さとノイズ量を0～1の画像状態スコアとして推定する。"""

    pixels = green[mask > 0]

    if pixels.size == 0:
        return {
            "mean": 0.0,
            "median": 0.0,
            "p10": 0.0,
            "p90": 0.0,
            "contrast": 0.0,
            "noise_std": 0.0,
            "noise_ratio": 0.0,
            "condition": 0.0,
        }

    mean_value = float(np.mean(pixels))
    median_value = float(np.median(pixels))
    p10 = float(np.percentile(pixels, 10.0))
    p90 = float(np.percentile(pixels, 90.0))
    contrast = max(p90 - p10, 1.0)

    noise_std = estimate_green_noise(green, mask)
    noise_ratio = noise_std / contrast

    dark_mean_score = np.clip(
        (ADAPTIVE_DARK_MEAN - mean_value) / 45.0,
        0.0, 1.0,
    )
    dark_median_score = np.clip(
        (ADAPTIVE_DARK_MEDIAN - median_value) / 45.0,
        0.0, 1.0,
    )
    noise_score_abs = np.clip(
        (noise_std - ADAPTIVE_NOISE_STD) / 12.0,
        0.0, 1.0,
    )
    noise_score_ratio = np.clip(
        (noise_ratio - ADAPTIVE_NOISE_RATIO) / 0.10,
        0.0, 1.0,
    )

    condition = float(np.clip(
        max(
            dark_mean_score,
            dark_median_score,
            noise_score_abs,
            noise_score_ratio,
        ),
        0.0, 1.0,
    ))

    return {
        "mean": mean_value,
        "median": median_value,
        "p10": p10,
        "p90": p90,
        "contrast": contrast,
        "noise_std": noise_std,
        "noise_ratio": noise_ratio,
        "condition": condition,
    }


def remove_radial_banding(
    green,
    mask,
    condition,
):
    """
    Fundus中心を基準とした同心円状の階調バンディングを抑える。

    半径ごとの平均プロファイルを推定し、その中の
    「急激に変化する半径方向成分」だけを減算する。
    血管は角度方向に局所的な構造なので、半径全体の平均には
    入りにくい。
    """
    if not RADIAL_BANDING_ENABLED:
        return green.copy(), np.zeros_like(green, dtype=np.float32)

    condition = float(np.clip(condition, 0.0, 1.0))
    if condition < RADIAL_BANDING_MIN_CONDITION:
        return green.copy(), np.zeros_like(green, dtype=np.float32)

    valid = mask > 0
    ys, xs = np.nonzero(valid)
    if len(xs) < 100:
        return green.copy(), np.zeros_like(green, dtype=np.float32)

    h, w = green.shape
    cx = float(np.mean(xs))
    cy = float(np.mean(ys))

    y_grid, x_grid = np.mgrid[0:h, 0:w].astype(np.float32)
    radius = np.sqrt(
        (x_grid - cx) ** 2 +
        (y_grid - cy) ** 2
    )
    radius_int = np.rint(radius).astype(np.int32)

    max_radius = int(np.max(radius_int[valid]))
    values = green.astype(np.float32)

    sums = np.bincount(
        radius_int[valid].ravel(),
        weights=values[valid].ravel(),
        minlength=max_radius + 1,
    )
    counts = np.bincount(
        radius_int[valid].ravel(),
        minlength=max_radius + 1,
    ).astype(np.float32)

    # 平均では血管の影響を受けやすいため、まずbin平均を作り、
    # 後段で強く平滑化して「ゆっくり変化する円周背景」だけを残す。
    profile = sums / np.maximum(counts, 1.0)

    valid_profile = counts > 0
    if np.count_nonzero(valid_profile) < 10:
        return green.copy(), np.zeros_like(green, dtype=np.float32)

    indices = np.arange(len(profile), dtype=np.float32)
    known_x = indices[valid_profile]
    known_y = profile[valid_profile]
    profile = np.interp(indices, known_x, known_y)

    smooth_radius = max(1, int(RADIAL_PROFILE_SMOOTH_RADIUS))
    kernel_x = np.arange(
        -smooth_radius,
        smooth_radius + 1,
        dtype=np.float32,
    )
    sigma = max(smooth_radius / 2.0, 1.0)
    kernel = np.exp(-(kernel_x ** 2) / (2.0 * sigma ** 2))
    kernel /= np.sum(kernel)

    smooth_profile = np.convolve(
        profile,
        kernel,
        mode="same",
    )

    radial_background = smooth_profile[
        np.clip(radius_int, 0, len(smooth_profile) - 1)
    ]

    # smooth_profileは「ゆっくり変化する本来の照明ムラ」。
    # profileとの差だけを補正することで、通常の照明勾配を
    # 消しすぎず、細かな同心円バンディングだけを狙う。
    correction = (
        profile - smooth_profile
    )[
        np.clip(radius_int, 0, len(smooth_profile) - 1)
    ]

    # conditionに応じて補正量を徐々に強くする。
    strength = np.clip(
        (condition - RADIAL_BANDING_MIN_CONDITION)
        / max(1.0 - RADIAL_BANDING_MIN_CONDITION, 1e-6),
        0.0,
        1.0,
    ) * RADIAL_BANDING_STRENGTH

    corrected = (
        values - correction * strength
    )

    # 元画像から大きく外れないよう、変化量を軽く制限。
    corrected = np.clip(
        corrected,
        0.0,
        255.0,
    ).astype(np.uint8)

    corrected = mask_image(
        corrected,
        mask,
    )

    banding_visual = np.clip(
        np.abs(correction * strength) * 4.0,
        0.0,
        255.0,
    ).astype(np.uint8)
    banding_visual = mask_image(
        banding_visual,
        mask,
    )

    return corrected, banding_visual


def adaptive_smooth_green(
    green,
    mask,
    condition,
):
    """問題画像だけedge-preserving smoothingを追加する。"""

    if condition < ADAPTIVE_CONDITION_THRESHOLD:
        smoothed = cv2.GaussianBlur(
            green,
            BLUR_KERNEL_NORMAL,
            0,
        )
    else:
        strength = np.clip(
            (condition - ADAPTIVE_CONDITION_THRESHOLD)
            / max(1.0 - ADAPTIVE_CONDITION_THRESHOLD, 1e-6),
            0.0, 1.0,
        )
        sigma_color = (
            ADAPTIVE_BILATERAL_SIGMA_COLOR
            + 10.0 * strength
        )
        smoothed = cv2.bilateralFilter(
            green,
            d=ADAPTIVE_BILATERAL_D,
            sigmaColor=float(sigma_color),
            sigmaSpace=ADAPTIVE_BILATERAL_SIGMA_SPACE,
        )

    return mask_image(smoothed, mask)


# ============================================================
# 応答正規化
# ============================================================

# ============================================================
# 応答マップの正規化パーセンタイル
#
# 視神経乳頭の縁など、局所的に極端に強い応答が
# ごく少数あると、そこに引っ張られて0-255への
# ストレッチ幅が広がりすぎ、本来の血管信号
# （相対的に弱い）が0-255の狭い範囲に押し込まれて
# 8bit量子化時に情報が潰れてしまう。
#
# 上限percentileを下げることで、少数の極端な外れ値の
# 影響を抑え、血管信号が実際に使われている値域を
# 広く（＝量子化を細かく）保つ。
# ============================================================

RESPONSE_NORMALIZE_LOW_PERCENTILE = 1.0
RESPONSE_NORMALIZE_HIGH_PERCENTILE = 97.0


def normalize_response(
    response,
    mask,
):
    output = np.zeros_like(
        response,
        dtype=np.uint8,
    )

    pixels = response[
        mask > 0
    ]

    pixels = pixels[
        np.isfinite(pixels)
    ]

    pixels = pixels[
        pixels > 0
    ]

    if pixels.size == 0:
        return output

    percentile_1 = float(
        np.percentile(
            pixels,
            RESPONSE_NORMALIZE_LOW_PERCENTILE,
        )
    )

    percentile_995 = float(
        np.percentile(
            pixels,
            RESPONSE_NORMALIZE_HIGH_PERCENTILE,
        )
    )

    if percentile_995 <= percentile_1 + 1e-8:
        percentile_995 = float(
            np.max(pixels)
        )

    if percentile_995 <= percentile_1 + 1e-8:
        return output

    normalized = (
        response.astype(np.float32)
        - percentile_1
    ) / (
        percentile_995
        - percentile_1
    )

    normalized = np.clip(
        normalized,
        0.0,
        1.0,
    )

    output = (
        normalized * 255.0
    ).astype(np.uint8)

    return mask_image(
        output,
        mask,
    )


# ============================================================
# Mask境界付近の応答抑制
# ============================================================

def suppress_near_mask_border(
    response,
    mask,
    border_width=BORDER_SUPPRESSION_WIDTH,
):
    valid = mask > 0

    if not np.any(valid):
        return np.zeros_like(
            response,
            dtype=np.uint8,
        )

    if border_width <= 0:
        return mask_image(
            response,
            mask,
        )

    pad = int(border_width) + 2

    padded_mask = cv2.copyMakeBorder(
        mask,
        pad,
        pad,
        pad,
        pad,
        borderType=cv2.BORDER_CONSTANT,
        value=0,
    )

    padded_distance = cv2.distanceTransform(
        padded_mask,
        cv2.DIST_L2,
        5,
    )

    distance = padded_distance[
        pad:pad + mask.shape[0],
        pad:pad + mask.shape[1],
    ]

    falloff = np.clip(
        distance.astype(np.float32)
        / max(
            float(border_width),
            1.0,
        ),
        0.0,
        1.0,
    )

    result = (
        response.astype(np.float32)
        * falloff
    )

    result[~valid] = 0.0

    return np.clip(
        result,
        0,
        255,
    ).astype(np.uint8)


# ============================================================
# Frangi応答
# ============================================================

def build_frangi_response(
    green,
    mask,
    condition=0.0,
    ring_score=None,
    use_preprocessed_direct=False,
):
    """
    暗い血管に対するマルチスケールFrangi応答。

    暗い画像ではCLAHEを強くかけるほど眼底テクスチャまで
    強調されるため、画像状態に応じてCLAHEを弱める。
    特に暗い画像では0.7pxのfine responseをさらに抑え、
    1px以上の安定した管状構造を優先する。
    """
    condition = float(np.clip(condition, 0.0, 1.0))

    # ========================================================
    # 重要：前処理済み画像を直接使用
    # ========================================================
    # fundus_preprocessing.pyですでに
    #   ・暗画像の事前denoise
    #   ・illumination correction
    #   ・brightness adjustment
    #   ・CLAHE
    #   ・最終denoise
    # が済んでいる。
    #
    # ここで再度normalize + CLAHEをすると、暗い画像では
    # 脈絡膜の微細模様まで再び強調され、Frangiの偽陽性を
    # 増やす原因になる。そのため直接入力モードでは
    # 前処理済み8bit画像をそのまま使う。
    # ========================================================
    if use_preprocessed_direct:
        green_normalized = np.clip(
            green,
            0,
            255,
        ).astype(np.uint8)
    else:
        green_normalized = normalize_inside_mask(green, mask)

        clahe_clip = (
            CLAHE_CLIP_NORMAL
            - (CLAHE_CLIP_NORMAL - CLAHE_CLIP_DARK) * condition
        )

        if condition >= ADAPTIVE_CONDITION_THRESHOLD:
            clahe_clip = min(clahe_clip, 0.80)

        clahe = cv2.createCLAHE(
            clipLimit=float(clahe_clip),
            tileGridSize=(8, 8),
        )

        clahe_image = clahe.apply(green_normalized)

        if condition >= ADAPTIVE_CONDITION_THRESHOLD:
            blend = 0.25 + 0.25 * (1.0 - condition)
            green_normalized = cv2.addWeighted(
                green_normalized,
                1.0 - blend,
                clahe_image,
                blend,
                0.0,
            )
        else:
            green_normalized = clahe_image

    green_normalized = mask_image(
        green_normalized,
        mask,
    )

    image = (
        fill_outside_with_median(
            green_normalized,
            mask,
        )
        / 255.0
    ).astype(np.float32)

    main_sigmas = FRANGI_SIGMAS[1:]

    main_response = frangi(
        image,
        sigmas=main_sigmas,
        alpha=FRANGI_ALPHA,
        beta=FRANGI_BETA,
        gamma=FRANGI_GAMMA,
        black_ridges=True,
    )

    fine_response = frangi(
        image,
        sigmas=(FRANGI_SIGMAS[0],),
        alpha=FRANGI_ALPHA,
        beta=FRANGI_BETA,
        gamma=FRANGI_GAMMA,
        black_ridges=True,
    )

    main_response = np.nan_to_num(
        main_response.astype(np.float32),
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    fine_response = np.nan_to_num(
        fine_response.astype(np.float32),
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    fine_weight = (
        FINE_FRANGI_WEIGHT
        - 0.25 * condition
    )
    fine_weight = float(np.clip(fine_weight, 0.45, 0.82))

    response = np.maximum(
        main_response,
        fine_response * fine_weight,
    )

    response[mask == 0] = 0.0

    normalized = normalize_response(
        response,
        mask,
    )

    normalized = suppress_near_mask_border(
        normalized,
        mask,
    )

    return normalized, green_normalized


# ============================================================
# 局所コントラスト正規化（新規）
# ============================================================

def build_local_relative_frangi(
    frangi_response,
    mask,
    radius=LOCAL_CONTRAST_RADIUS,
):
    """
    Frangi応答を「局所平均に対する比率」として捉え直す。

    問題意識：

        画像全体で同じpercentileを使うと、
        もともとコントラストが低い周辺部の細い血管は
        いつも不利になる。

        かといって画像全体の閾値を下げると、
        今度は別の場所（視神経乳頭周辺やノイズが
        多い領域）で誤検出が増える。

    そこで、応答をその画素の周辺（局所領域）の
    平均応答で割った「比率」として評価し直す。

    これにより、
        「画像全体で見て強いかどうか」ではなく
        「その場所の平均に対して、どれだけ
         突出した線状構造か」
    という基準になり、画像全体のコントラストの
    ムラに影響されにくくなる。

    ノイズ対策：
        この比率だけで採用すると誤検出が増えるため、
        実際の採用時には必ず

            ・方向一貫性(coherence)
            ・暗線確認(dark)
            ・strong seedからのgeodesic hysteresis

        を組み合わせて使うこと。単独では使わない。
    """

    response_f = frangi_response.astype(np.float32)
    mask_f = (mask > 0).astype(np.float32)

    masked_response = response_f * mask_f

    local_sum = cv2.GaussianBlur(
        masked_response,
        (0, 0),
        sigmaX=radius,
        sigmaY=radius,
    )

    local_count = cv2.GaussianBlur(
        mask_f,
        (0, 0),
        sigmaX=radius,
        sigmaY=radius,
    )

    local_mean = local_sum / np.maximum(
        local_count,
        1e-3,
    )

    # 局所平均が極端に小さい（ほぼ応答がない）場所で
    # 比率が発散しないよう下限を設ける。
    local_mean = np.maximum(
        local_mean,
        3.0,
    )

    ratio = response_f / local_mean

    ratio[mask == 0] = 0.0

    return ratio


# ============================================================
# Green画像の局所平均による暗線候補
# ============================================================

def build_local_block_darkness(
    green,
    mask,
    grid_size=LOCAL_DARK_GRID_SIZE,
):
    """
    Green画像をgrid_size x grid_sizeに分割し、各セルから
    「血管や極端な暗部を除いた背景輝度」を推定する。

    単純なセル平均では、セル内に暗い血管が多いと背景平均まで
    下がってしまい、周囲の模様を大量に「暗い線」と誤判定する。
    そこで各セルについて下位15%と上位15%を除いたtrimmed meanを
    背景平均として使用する。

    さらに局所標準偏差からz-like darknessを作り、

        darkness = local_background - pixel

    が十分大きい場所だけを極薄血管候補とする。
    """
    image = green.astype(np.float32)
    valid = mask > 0

    if not np.any(valid):
        return (
            np.zeros_like(image, dtype=np.float32),
            np.zeros_like(image, dtype=np.float32),
            np.zeros_like(image, dtype=np.uint8),
        )

    h, w = image.shape
    rows = max(1, int(grid_size))
    cols = max(1, int(grid_size))

    global_pixels = image[valid]
    global_mean = float(np.mean(global_pixels))
    global_std = float(np.std(global_pixels))

    mean_grid = np.zeros((rows, cols), dtype=np.float32)
    std_grid = np.zeros((rows, cols), dtype=np.float32)

    for gy in range(rows):
        y0 = (gy * h) // rows
        y1 = ((gy + 1) * h) // rows

        for gx in range(cols):
            x0 = (gx * w) // cols
            x1 = ((gx + 1) * w) // cols

            cell_mask = valid[y0:y1, x0:x1]
            cell_pixels = image[y0:y1, x0:x1][cell_mask]

            if cell_pixels.size > 30:
                p15 = float(np.percentile(cell_pixels, 15.0))
                p85 = float(np.percentile(cell_pixels, 85.0))

                trimmed = cell_pixels[
                    (cell_pixels >= p15)
                    & (cell_pixels <= p85)
                ]

                if trimmed.size > 10:
                    mean_grid[gy, gx] = float(np.mean(trimmed))
                    std_grid[gy, gx] = float(np.std(trimmed))
                else:
                    mean_grid[gy, gx] = float(np.mean(cell_pixels))
                    std_grid[gy, gx] = float(np.std(cell_pixels))
            else:
                mean_grid[gy, gx] = global_mean
                std_grid[gy, gx] = global_std

    local_mean = cv2.resize(
        mean_grid,
        (w, h),
        interpolation=cv2.INTER_LINEAR,
    )

    local_std = cv2.resize(
        std_grid,
        (w, h),
        interpolation=cv2.INTER_LINEAR,
    )

    local_std = np.maximum(local_std, 3.0)

    darkness = np.maximum(
        local_mean - image,
        0.0,
    )

    darkness_z = darkness / (
        local_std * max(LOCAL_DARK_STD_FACTOR, 0.10)
        + 1.0
    )

    local_dark_binary = (
        valid
        & (darkness >= LOCAL_DARK_MIN_DROP)
        & (darkness_z >= LOCAL_DARK_MIN_Z)
    )

    local_dark_binary = (
        local_dark_binary.astype(np.uint8)
        * 255
    )

    # マスク境界そのものは局所平均より暗くなりやすいため除外。
    border_distance = cv2.distanceTransform(
        np.where(mask > 0, 255, 0).astype(np.uint8),
        cv2.DIST_L2,
        5,
    )

    local_dark_binary[
        border_distance < float(BORDER_SUPPRESSION_WIDTH)
    ] = 0

    local_dark_binary = np.where(
        local_dark_binary > 0,
        255,
        0,
    ).astype(np.uint8)

    darkness[~valid] = 0.0
    darkness_z[~valid] = 0.0

    return (
        darkness,
        darkness_z,
        local_dark_binary,
    )

# ============================================================
# Black-hat応答
# ============================================================

def build_multiscale_blackhat_response(
    green_normalized,
    mask,
):
    image = fill_outside_with_median(
        green_normalized,
        mask,
    ).astype(np.uint8)

    response_max = np.zeros_like(
        image,
        dtype=np.float32,
    )

    for kernel_size in BLACKHAT_KERNEL_SIZES:
        size = int(kernel_size)

        if size % 2 == 0:
            size += 1

        kernel = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (
                size,
                size,
            ),
        )

        blackhat = cv2.morphologyEx(
            image,
            cv2.MORPH_BLACKHAT,
            kernel,
        )

        response_max = np.maximum(
            response_max,
            blackhat.astype(np.float32),
        )

    response_max[mask == 0] = 0.0

    normalized = normalize_response(
        response_max,
        mask,
    )

    normalized = suppress_near_mask_border(
        normalized,
        mask,
    )

    return normalized


# ============================================================
# 局所暗線応答
# ============================================================

def build_dark_line_response(
    green_normalized,
    mask,
):
    image = fill_outside_with_median(
        green_normalized,
        mask,
    )

    background = cv2.GaussianBlur(
        image,
        (0, 0),
        sigmaX=DARK_BACKGROUND_SIGMA,
        sigmaY=DARK_BACKGROUND_SIGMA,
    )

    dark_response = np.maximum(
        background - image,
        0.0,
    )

    dark_response[mask == 0] = 0.0

    normalized = normalize_response(
        dark_response.astype(np.float32),
        mask,
    )

    return suppress_near_mask_border(
        normalized,
        mask,
    )


# ============================================================
# 構造テンソルによる方向一貫性
# ============================================================

def build_line_coherence(
    green_normalized,
    mask,
    condition=0.0,
):
    """
    構造テンソルから局所方向の一貫性を計算する。

    線状構造では値が高くなりやすく、
    等方的な粒状ノイズや斑点では低くなりやすい。

    戻り値は0～255。
    """

    image = fill_outside_with_median(
        green_normalized,
        mask,
    ).astype(np.float32)

    image /= 255.0

    gradient_x = cv2.Sobel(
        image,
        cv2.CV_32F,
        1,
        0,
        ksize=3,
    )

    gradient_y = cv2.Sobel(
        image,
        cv2.CV_32F,
        0,
        1,
        ksize=3,
    )

    condition = float(np.clip(condition, 0.0, 1.0))
    coherence_sigma = (
        COHERENCE_SIGMA_NORMAL
        + (COHERENCE_SIGMA_DARK - COHERENCE_SIGMA_NORMAL) * condition
    )

    tensor_xx = cv2.GaussianBlur(
        gradient_x * gradient_x,
        (0, 0),
        sigmaX=coherence_sigma,
        sigmaY=coherence_sigma,
    )

    tensor_yy = cv2.GaussianBlur(
        gradient_y * gradient_y,
        (0, 0),
        sigmaX=coherence_sigma,
        sigmaY=coherence_sigma,
    )

    tensor_xy = cv2.GaussianBlur(
        gradient_x * gradient_y,
        (0, 0),
        sigmaX=coherence_sigma,
        sigmaY=coherence_sigma,
    )

    discriminant = np.sqrt(
        np.maximum(
            (
                tensor_xx
                - tensor_yy
            ) ** 2
            + 4.0 * tensor_xy ** 2,
            0.0,
        )
    )

    lambda_1 = 0.5 * (
        tensor_xx
        + tensor_yy
        + discriminant
    )

    lambda_2 = 0.5 * (
        tensor_xx
        + tensor_yy
        - discriminant
    )

    coherence = (
        lambda_1
        - lambda_2
    ) / (
        lambda_1
        + lambda_2
        + 1e-6
    )

    coherence = np.clip(
        coherence,
        0.0,
        1.0,
    )

    coherence[mask == 0] = 0.0

    coherence_uint8 = (
        coherence * 255.0
    ).astype(np.uint8)

    return suppress_near_mask_border(
        coherence_uint8,
        mask,
    )


# ============================================================
# Frangiノイズ比率
# ============================================================

def estimate_frangi_noise_ratio(
    frangi_response,
    mask,
):
    """
    Otsu法で強い応答に分類される画素の割合を返す。
    """

    valid = (
        (mask > 0)
        & (frangi_response > 0)
    )

    pixels = frangi_response[
        valid
    ]

    if pixels.size == 0:
        return 0.0, 0

    otsu_value, _ = cv2.threshold(
        pixels.reshape(-1, 1),
        0,
        255,
        cv2.THRESH_BINARY
        + cv2.THRESH_OTSU,
    )

    ratio = float(
        np.mean(
            pixels >= otsu_value
        )
    )

    return (
        ratio,
        int(otsu_value),
    )


# ============================================================
# Geodesic hysteresis
# ============================================================

def geodesic_hysteresis(
    strong,
    candidate,
    mask,
):
    """
    strong seedからcandidate内だけを反復的に成長させる。

    candidate内に存在しても、strong seedに接続していない
   独立ノイズ成分は採用しない。
    """

    strong_binary = np.where(
        (strong > 0)
        & (mask > 0),
        255,
        0,
    ).astype(np.uint8)

    candidate_binary = np.where(
        (candidate > 0)
        & (mask > 0),
        255,
        0,
    ).astype(np.uint8)

    # strongは必ずcandidateに含める。
    candidate_binary[
        strong_binary > 0
    ] = 255

    result = strong_binary.copy()

    kernel_size = HYSTERESIS_KERNEL_SIZE

    if kernel_size % 2 == 0:
        kernel_size += 1

    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (
            kernel_size,
            kernel_size,
        ),
    )

    for _ in range(HYSTERESIS_MAX_ITERATIONS):
        expanded = cv2.dilate(
            result,
            kernel,
            iterations=1,
        )

        expanded = cv2.bitwise_and(
            expanded,
            candidate_binary,
        )

        if np.array_equal(
            expanded,
            result,
        ):
            break

        result = expanded

    return mask_image(
        result,
        mask,
    )


# ============================================================
# 血管候補threshold
# ============================================================

def build_ring_artifact_score(
    green_normalized,
    mask,
):
    """
    Fundus中心を基準に、円周方向の弱い線状構造を0～1で推定する。

    同心円ノイズでは「線の法線方向」と「中心から外向きの方向」が
    近くなる。そこでgradient方向とradial方向の一致度を使う。
    血管の強さそのものではなく、あくまで暗い画像の弱い候補を
    抑えるための補助マップとして使用する。
    """
    valid = mask > 0
    h, w = mask.shape

    ys, xs = np.nonzero(valid)
    if len(xs) < 100:
        return np.zeros_like(green_normalized, dtype=np.float32)

    cx = float(np.mean(xs))
    cy = float(np.mean(ys))

    y_grid, x_grid = np.mgrid[0:h, 0:w].astype(np.float32)
    dx = x_grid - cx
    dy = y_grid - cy
    radius = np.sqrt(dx * dx + dy * dy)

    radial_x = dx / np.maximum(radius, 1e-6)
    radial_y = dy / np.maximum(radius, 1e-6)

    image = fill_outside_with_median(
        green_normalized,
        mask,
    ).astype(np.float32) / 255.0

    gx = cv2.Sobel(
        image,
        cv2.CV_32F,
        1,
        0,
        ksize=3,
    )
    gy = cv2.Sobel(
        image,
        cv2.CV_32F,
        0,
        1,
        ksize=3,
    )

    grad_mag = np.sqrt(gx * gx + gy * gy)

    # 円周状の線ではgradientがradial方向を向く。
    radial_alignment = np.abs(
        gx * radial_x + gy * radial_y
    ) / np.maximum(grad_mag, 1e-6)

    # 微弱なgradientだけでリング判定しない。
    grad_reference = np.percentile(
        grad_mag[valid],
        75.0,
    )
    grad_strength = np.clip(
        grad_mag / max(float(grad_reference), 1e-6),
        0.0,
        1.0,
    )

    score = radial_alignment * grad_strength

    # 中心付近とmask外は無効。
    fundus_radius = float(np.max(radius[valid]))
    radius_ratio = radius / max(fundus_radius, 1.0)

    score[
        (radius_ratio < RING_ARTIFACT_MIN_RADIUS_RATIO)
        | (radius_ratio > RING_ARTIFACT_MAX_RADIUS_RATIO)
        | (~valid)
    ] = 0.0

    return np.clip(
        score,
        0.0,
        1.0,
    ).astype(np.float32)


# ============================================================
# 血管候補threshold
# ============================================================

def threshold_vessels(
    frangi_response,
    blackhat,
    dark,
    coherence,
    local_relative,
    local_darkness_z,
    local_dark_binary,
    mask,
    condition=0.0,
    ring_score=None,
):
    """
    Frangiを主応答として血管候補を作る。

    重要なのは「strong seedを非常に厳しくする」こと。
    元コードではFrangiが高いだけでstrong seedになっていたため、
    眼底の鱗状・渦状テクスチャまでseedになり、
    geodesic hysteresisによって広い範囲が血管として残る可能性がある。

    今回は、

        strong seed
            = 高Frangi
            AND 暗線
            AND Black-hat
            AND 方向一貫性

    とする。

    一方、薄い血管はweak candidateとして広めに残し、
    strong seedにつながっている場合だけhysteresisで救済する。
    これにより「薄い血管を残す」と「ノイズをseedにしない」を両立する。
    """

    valid = (
        (mask > 0)
        & (frangi_response > 0)
    )

    frangi_pixels = frangi_response[
        valid
    ]

    if frangi_pixels.size == 0:
        return (
            np.zeros_like(
                frangi_response,
                dtype=np.uint8,
            ),
            0.0,
            0.0,
            0.0,
            np.zeros_like(
                frangi_response,
                dtype=np.uint8,
            ),
            np.zeros_like(
                frangi_response,
                dtype=np.uint8,
            ),
        )

    noise_ratio, otsu_value = (
        estimate_frangi_noise_ratio(
            frangi_response,
            mask,
        )
    )

    condition = float(np.clip(condition, 0.0, 1.0))

    if ring_score is None:
        ring_score = np.zeros_like(
            frangi_response,
            dtype=np.float32,
        )

    ring_bad = (
        RING_ARTIFACT_ENABLED
        & (condition >= RING_ARTIFACT_CONDITION_MIN)
        & (ring_score >= RING_ARTIFACT_SCORE_THRESHOLD)
    )

    high_percentile = FRANGI_HIGH_PERCENTILE
    low_percentile = FRANGI_LOW_PERCENTILE
    blackhat_percentile = BLACKHAT_SUPPORT_PERCENTILE
    dark_percentile = DARK_SUPPORT_PERCENTILE

    # 問題画像ではseedを少し厳しくし、candidateでは
    # Black-hatによる「本当に暗い線」の確認を重視する。
    high_percentile += 3.0 * condition
    low_percentile += 2.0 * condition
    blackhat_percentile += 2.0 * condition
    dark_percentile += 1.0 * condition

    if noise_ratio > FRANGI_NOISE_RATIO_LIMIT:
        boost = min(
            (
                noise_ratio
                - FRANGI_NOISE_RATIO_LIMIT
            )
            * 20.0,
            FRANGI_PERCENTILE_MAX_BOOST,
        )

        high_percentile = min(
            96.0,
            high_percentile + boost,
        )

        low_percentile = min(
            92.0,
            low_percentile + boost,
        )

        blackhat_percentile = min(
            90.0,
            blackhat_percentile + boost,
        )

        dark_percentile = min(
            90.0,
            dark_percentile + boost,
        )

    high_threshold = float(
        np.percentile(
            frangi_pixels,
            high_percentile,
        )
    )

    low_threshold = float(
        np.percentile(
            frangi_pixels,
            low_percentile,
        )
    )

    low_threshold = max(
        low_threshold,
        high_threshold * 0.18,
        3.0,
    )

    if noise_ratio > FRANGI_NOISE_RATIO_LIMIT:
        high_threshold = max(
            high_threshold,
            min(
                float(otsu_value),
                high_threshold + 10.0,
            ),
        )

        low_threshold = max(
            low_threshold,
            high_threshold * 0.18,
        )

    blackhat_pixels = blackhat[
        mask > 0
    ]
    blackhat_pixels = blackhat_pixels[
        blackhat_pixels > 0
    ]

    if blackhat_pixels.size > 0:
        blackhat_threshold = float(
            np.percentile(
                blackhat_pixels,
                blackhat_percentile,
            )
        )
    else:
        blackhat_threshold = 256.0

    dark_pixels = dark[
        mask > 0
    ]
    dark_pixels = dark_pixels[
        dark_pixels > 0
    ]

    if dark_pixels.size > 0:
        dark_threshold = float(
            np.percentile(
                dark_pixels,
                dark_percentile,
            )
        )
    else:
        dark_threshold = 256.0

    coherence_threshold = int(
        np.clip(
            COHERENCE_MIN,
            0.0,
            1.0,
        )
        * 255.0
    )

    coherence_supported = (
        coherence
        >= coherence_threshold
    )

    blackhat_supported = (
        blackhat
        >= blackhat_threshold
    )

    dark_supported = (
        dark
        >= dark_threshold
    )

    strong_frangi = (
        (frangi_response >= high_threshold)
        & (mask > 0)
    )

    weak_frangi = (
        (frangi_response >= low_threshold)
        & (mask > 0)
    )

    medium_frangi = (
        frangi_response
        >= high_threshold * 0.50
    )

    # --------------------------------------------------------
    # 強いseed
    #
    # Frangiだけではなく、
    #   ・暗い線
    #   ・Black-hat
    #   ・方向一貫性
    # の3条件を同時に要求する。
    #
    # ここを厳しくすることで、鱗状ノイズが
    # hysteresisの起点になるのを防ぐ。
    # --------------------------------------------------------

    strong = (
        strong_frangi
        & dark_supported
        & blackhat_supported
        & coherence_supported
    )

    # 同心円状アーティファクトは暗い画像でのみ、弱いseedから除外。
    # 強い本物の主幹血管を一律に消さない。
    strong &= ~ring_bad


    # --------------------------------------------------------
    # 弱い血管候補
    #
    # thin vesselはstrongにならなくてもよい。
    # ただし単なるFrangiだけではなく、
    # 暗線 + 方向一貫性を基本にする。
    # --------------------------------------------------------

    if condition >= ADAPTIVE_CONDITION_THRESHOLD:
        # 暗い/ノイズの多い画像では、medium Frangiだけを
        # 許可するとテクスチャがcandidateへ大量に入る。
        supported_weak_global = (
            weak_frangi
            & dark_supported
            & coherence_supported
            & blackhat_supported
        )
    else:
        supported_weak_global = (
            weak_frangi
            & dark_supported
            & coherence_supported
            & (
                medium_frangi
                | blackhat_supported
            )
        )

    # --------------------------------------------------------
    # 局所コントラストによる細い血管救済
    #
    # グローバルには弱くても、その場所の周囲より
    # 明らかに線状応答が強い場合を救済する。
    #
    # local_relativeだけではノイズも大量に拾うため、
    # 暗線 + coherenceを必須とし、
    # Black-hatまたはmedium Frangiも要求する。
    # --------------------------------------------------------

    local_ratio_threshold = (
        LOCAL_CONTRAST_RATIO_THRESHOLD
        + 0.35 * condition
    )

    locally_prominent = (
        (local_relative >= local_ratio_threshold)
        & (frangi_response >= LOCAL_CONTRAST_MIN_FRANGI)
    )

    if condition >= ADAPTIVE_CONDITION_THRESHOLD:
        supported_weak_local = (
            locally_prominent
            & dark_supported
            & coherence_supported
            & blackhat_supported
        )
    else:
        supported_weak_local = (
            locally_prominent
            & dark_supported
            & coherence_supported
            & (
                blackhat_supported
                | medium_frangi
            )
        )

    # --------------------------------------------------------
    # Green画像の局所平均による極薄血管救済
    #
    # ここでは「局所平均より暗い」だけでは採用しない。
    # mask境界を除外したうえで、少なくとも
    #   1. 局所的に暗い
    #   2. 方向性がある
    #   3. Frangi / dark / Black-hatのいずれかが弱くても反応
    # を要求する。
    #
    # これにより、Frangiのglobal percentileでは落ちた
    # 極端に薄い血管を候補へ戻す。
    # --------------------------------------------------------

    local_dark_support_values = local_darkness_z[mask > 0]
    local_dark_support_values = local_dark_support_values[
        np.isfinite(local_dark_support_values)
        & (local_dark_support_values > 0)
    ]

    if local_dark_support_values.size > 0:
        local_dark_support_threshold = float(
            np.percentile(
                local_dark_support_values,
                LOCAL_DARK_SUPPORT_PERCENTILE,
            )
        )
        local_dark_support_threshold = max(
            local_dark_support_threshold,
            LOCAL_DARK_MIN_Z,
        )
    else:
        local_dark_support_threshold = LOCAL_DARK_MIN_Z

    # 極薄血管救済は「暗い」だけでは採用しない。
    # 局所的な暗さ + 長い方向性 + 少なくとも1つの線状応答を要求する。
    # これにより、暗い画像の脈絡膜テクスチャを大量に拾うのを防ぐ。
    local_dark_line = (
        (local_darkness_z >= local_dark_support_threshold)
        & (local_dark_binary > 0)
        & (coherence >= LOCAL_DARK_MIN_COHERENCE * 255.0)
        & (
            (frangi_response >= LOCAL_DARK_MIN_FRANGI)
            | (dark >= 10.0)
            | (blackhat >= 10.0)
        )
        & (mask > 0)
    )

    # Frangiが極端に弱い末端を救済する場合でも、
    # dark + blackhat の両方、またはFrangiを最低限要求する。
    local_dark_dual_support = (
        (local_darkness_z >= LOCAL_DARK_MIN_Z)
        & (local_dark_binary > 0)
        & (coherence >= (LOCAL_DARK_MIN_COHERENCE + 0.02) * 255.0)
        & (
            (
                (dark >= 12.0)
                & (blackhat >= 10.0)
            )
            | (frangi_response >= 2.5)
        )
        & (mask > 0)
    )

    local_dark_candidate = (
        local_dark_line
        | local_dark_dual_support
    )

    # 極端に暗い画像では局所平均だけによる救済をさらに制限する。
    # 暗画像のテクスチャは局所的には血管と同じように暗く見えるため、
    # Frangiまたはdark/blackhatの線状応答が明確な画素だけを残す。
    if condition >= 0.55:
        local_dark_candidate &= (
            (frangi_response >= 4.0)
            | ((dark >= 14.0) & (blackhat >= 12.0))
        )

    supported_weak = (
        supported_weak_global
        | supported_weak_local
        | local_dark_candidate
    )

    # 眼底マスクの境界線そのものを血管として採用しない。
    border_distance = cv2.distanceTransform(
        np.where(mask > 0, 255, 0).astype(np.uint8),
        cv2.DIST_L2,
        5,
    )
    border_safe = border_distance >= float(BORDER_SUPPRESSION_WIDTH)
    supported_weak &= border_safe
    strong &= border_safe

    candidate = (
        supported_weak
        | strong
    )

    # ring_badの場所はcandidateからも除外するが、
    # 十分強いFrangi応答は残す。
    strong_enough_to_keep = (
        frangi_response >= high_threshold * RING_ARTIFACT_SUPPRESS_WEAK_FACTOR
    )

    candidate &= ~(
        ring_bad
        & ~strong_enough_to_keep
    )


    candidate_image = (
        candidate.astype(np.uint8)
        * 255
    )

    strong_image = (
        strong.astype(np.uint8)
        * 255
    )

    # 3x3 closeは近接したノイズ同士まで接続する可能性があるため、
    # ここではcloseを行わない。
    candidate_image = mask_image(
        candidate_image,
        mask,
    )

    result = geodesic_hysteresis(
        strong_image,
        candidate_image,
        mask,
    )

    result[strong] = 255

    result = mask_image(
        result,
        mask,
    )

    return (
        result,
        high_threshold,
        low_threshold,
        noise_ratio,
        strong_image,
        candidate_image,
    )



# ============================================================
# Strong seedからの血管成長
# ============================================================

def _skeleton_endpoints(skeleton):
    """8近傍数が1以下の骨格画素を端点として返す。"""
    binary = skeleton > 0
    if not np.any(binary):
        return []

    padded = np.pad(binary.astype(np.uint8), 1, mode="constant")
    neighbor_count = np.zeros_like(binary, dtype=np.uint8)

    for dy in (-1, 0, 1):
        for dx in (-1, 0, 1):
            if dy == 0 and dx == 0:
                continue
            neighbor_count += padded[
                1 + dy:1 + dy + binary.shape[0],
                1 + dx:1 + dx + binary.shape[1],
            ]

    ys, xs = np.where(binary & (neighbor_count <= 1))
    return list(zip(ys, xs))


def _estimate_tip_direction(skeleton, endpoint, lookback=10):
    """
    骨格端点から内側へ向かう局所方向をPCAで推定し、
    その逆向きを「これから血管が伸びる方向」とする。

    分岐点付近では完全な方向推定が難しいため、
    endpointから近い骨格画素だけを使用する。
    """
    y0, x0 = endpoint
    ys, xs = np.where(skeleton > 0)

    if len(xs) < 3:
        return None

    distance2 = (
        (xs.astype(np.float32) - float(x0)) ** 2
        + (ys.astype(np.float32) - float(y0)) ** 2
    )

    radius2 = float(max(lookback, 2) ** 2)
    local = distance2 <= radius2

    points = np.column_stack((
        xs[local].astype(np.float32),
        ys[local].astype(np.float32),
    ))

    if len(points) < 3:
        return None

    center = np.mean(points, axis=0)
    centered = points - center

    covariance = np.cov(centered.T)
    if covariance.shape != (2, 2):
        return None

    eigenvalues, eigenvectors = np.linalg.eigh(covariance)
    direction = eigenvectors[:, int(np.argmax(eigenvalues))]

    norm = float(np.linalg.norm(direction))
    if norm < 1e-6:
        return None

    direction = direction / norm

    endpoint_vector = np.array(
        [float(x0), float(y0)],
        dtype=np.float32,
    ) - center

    # PCA方向の符号を、endpointから外へ向くように合わせる。
    if float(np.dot(direction, endpoint_vector)) < 0.0:
        direction = -direction

    return direction.astype(np.float32)


def _candidate_score(
    y,
    x,
    direction,
    tip,
    frangi_response,
    blackhat,
    dark,
    coherence,
    local_relative,
    f_thr,
    d_thr,
    b_thr,
):
    """端点からの方向・血管特徴をまとめた成長スコア。"""
    eps = 1e-6

    point = np.array(
        [float(x), float(y)],
        dtype=np.float32,
    )
    tip_point = np.array(
        [float(tip[1]), float(tip[0])],
        dtype=np.float32,
    )

    step = point - tip_point
    step_norm = float(np.linalg.norm(step))
    if step_norm < eps:
        return -1e9, 0.0

    step_direction = step / step_norm
    alignment = abs(float(np.dot(direction, step_direction)))

    # 「前方」の点だけを許可するため、逆向きは強く減点する。
    forward_alignment = float(np.dot(direction, step_direction))
    if forward_alignment < SEED_GROWTH_DIRECTION_COSINE:
        return -1e9, forward_alignment

    f_score = float(frangi_response[y, x]) / max(f_thr, 1.0)
    d_score = float(dark[y, x]) / max(d_thr, 1.0)
    b_score = float(blackhat[y, x]) / max(b_thr, 1.0)
    c_score = float(coherence[y, x]) / 255.0
    local_score = min(
        float(local_relative[y, x]) / max(SEED_GROWTH_LOCAL_RATIO_MIN, 1.0),
        2.0,
    )

    # Frangiを中心に、dark/blackhat/coherenceを加点。
    # local_relativeは補助に留め、これ単独では高得点にならない。
    score = (
        0.48 * min(f_score, 2.0)
        + 0.18 * min(d_score, 2.0)
        + 0.14 * min(b_score, 2.0)
        + 0.14 * c_score
        + 0.06 * local_score
    )

    # 方向がずれるほど減点。
    score *= max(0.0, forward_alignment)

    return float(score), forward_alignment


def seed_guided_vessel_growth(
    strong_seed,
    base_vessel,
    candidate,
    frangi_response,
    blackhat,
    dark,
    coherence,
    local_relative,
    ring_score,
    mask,
    condition=0.0,
):
    """
    strong seed / 既存血管の端点から、弱い血管を「線の方向」に沿って
    少しずつ伸ばす。

    旧版の問題:
        accepted全体をdilateし、candidateと交差した場所を採用していた。
        そのため同心円状テクスチャがcandidateに含まれると、
        方向に関係なく横へ広がっていた。

    新版:
        1. 現在の骨格の端点だけを調べる
        2. 端点付近の骨格から進行方向を推定
        3. その方向の前方だけを探索
        4. Frangi + dark + blackhat + coherenceを複合評価
        5. 1回に1～3px程度しか進まない
        6. 採用した先を新しい端点として次の反復へ進む

    これにより、暗い画像の「独立したリング模様」を一括採用せず、
    本物の血管ネットワークだけをseedから連続的に救済する。
    """

    if not SEED_GROWTH_ENABLED:
        return (
            base_vessel.copy(),
            np.zeros_like(base_vessel, dtype=np.uint8),
        )

    valid = mask > 0
    if not np.any(valid):
        return (
            base_vessel.copy(),
            np.zeros_like(base_vessel, dtype=np.uint8),
        )

    strong_bool = (strong_seed > 0) & valid
    base_bool = (base_vessel > 0) & valid

    if not np.any(strong_bool) and not np.any(base_bool):
        return (
            base_vessel.copy(),
            np.zeros_like(base_vessel, dtype=np.uint8),
        )

    condition = float(np.clip(condition, 0.0, 1.0))

    def positive_percentile(image, percentile):
        values = image[
            valid
            & np.isfinite(image)
            & (image > 0)
        ]
        if values.size == 0:
            return 256.0
        return float(np.percentile(values, percentile))

    # 暗い画像ほど少しだけ厳しくする。
    f_percentile = min(
        62.0,
        SEED_GROWTH_FRANGI_PERCENTILE + 7.0 * condition,
    )
    d_percentile = min(
        64.0,
        SEED_GROWTH_DARK_PERCENTILE + 7.0 * condition,
    )
    b_percentile = min(
        68.0,
        SEED_GROWTH_BLACKHAT_PERCENTILE + 7.0 * condition,
    )

    f_thr = positive_percentile(frangi_response, f_percentile)
    d_thr = positive_percentile(dark, d_percentile)
    b_thr = positive_percentile(blackhat, b_percentile)

    coherence_thr = (
        SEED_GROWTH_COHERENCE_MIN
        + 0.05 * condition
    ) * 255.0

    local_thr = (
        SEED_GROWTH_LOCAL_RATIO_MIN
        + 0.10 * condition
    )

    # 成長候補。ここでもring_badを最初から除外する。
    ring_bad = np.zeros_like(valid, dtype=bool)
    if ring_score is not None:
        ring_bad = (
            RING_ARTIFACT_ENABLED
            & (condition >= RING_ARTIFACT_CONDITION_MIN)
            & (ring_score >= RING_ARTIFACT_SCORE_THRESHOLD)
        )

    relaxed = (
        (frangi_response >= f_thr)
        & (coherence >= coherence_thr)
        & (
            (dark >= d_thr)
            | (blackhat >= b_thr)
            | (local_relative >= local_thr)
        )
        & valid
    )

    # Frangiが少し弱くても、dark + blackhat + coherenceが揃う場合を救済。
    dual_support = (
        (frangi_response >= f_thr * 0.82)
        & (coherence >= coherence_thr)
        & (dark >= d_thr)
        & (blackhat >= b_thr)
        & valid
    )

    relaxed |= dual_support

    # 既存candidateは「成長可能領域」として使用するが、
    # 独立candidateをそのまま結果にはしない。
    relaxed |= (candidate > 0)
    relaxed &= valid

    # リングは成長通路から除外。
    # 強いFrangiだけは残すが、端点追跡では方向制約もかかる。
    if np.any(ring_bad):
        strong_keep = frangi_response >= max(f_thr * 1.25, f_thr + 8.0)
        relaxed &= ~(ring_bad & ~strong_keep)

    accepted = np.where(
        (strong_bool | base_bool),
        255,
        0,
    ).astype(np.uint8)

    accepted = mask_image(accepted, mask)

    # growth_addedはstrong/base以外の新規画素だけ。
    original_base = accepted.copy()

    # closeは無効。closeによるリング接続を防ぐ。
    growth_added = np.zeros_like(
        accepted,
        dtype=np.uint8,
    )

    # 1回の反復で大量の端点を処理しすぎない。
    # 各端点につき最大1画素だけ追加することで、線としての連続性を保つ。
    for _ in range(int(SEED_GROWTH_MAX_ITERATIONS)):
        skeleton_bool = skeletonize(accepted > 0)
        endpoints = _skeleton_endpoints(skeleton_bool)

        if not endpoints:
            break

        additions = []

        for endpoint in endpoints:
            direction = _estimate_tip_direction(
                skeleton_bool,
                endpoint,
                lookback=SEED_GROWTH_TIP_LOOKBACK,
            )

            if direction is None:
                continue

            y0, x0 = endpoint
            best = None
            best_score = -1e9

            # 端点の前方にある候補だけを調べる。
            for dy in range(
                -SEED_GROWTH_SEARCH_RADIUS,
                SEED_GROWTH_SEARCH_RADIUS + 1,
            ):
                for dx in range(
                    -SEED_GROWTH_SEARCH_RADIUS,
                    SEED_GROWTH_SEARCH_RADIUS + 1,
                ):
                    if dx == 0 and dy == 0:
                        continue

                    distance = float(np.hypot(dx, dy))
                    if distance > float(SEED_GROWTH_SEARCH_RADIUS):
                        continue

                    y = y0 + dy
                    x = x0 + dx

                    if (
                        y < 0
                        or y >= accepted.shape[0]
                        or x < 0
                        or x >= accepted.shape[1]
                    ):
                        continue

                    if not valid[y, x]:
                        continue

                    if accepted[y, x] > 0:
                        continue

                    if not relaxed[y, x]:
                        continue

                    score, forward_alignment = _candidate_score(
                        y,
                        x,
                        direction,
                        endpoint,
                        frangi_response,
                        blackhat,
                        dark,
                        coherence,
                        local_relative,
                        f_thr,
                        d_thr,
                        b_thr,
                    )

                    if score < SEED_GROWTH_MIN_SCORE:
                        continue

                    # 近傍に別の候補が1つ以上ある場合を優先。
                    neighborhood = relaxed[
                        max(0, y - 1):min(relaxed.shape[0], y + 2),
                        max(0, x - 1):min(relaxed.shape[1], x + 2),
                    ]
                    support_count = int(np.count_nonzero(neighborhood))

                    if support_count < SEED_GROWTH_MIN_SUPPORT_NEIGHBORS:
                        continue

                    # 距離が大きい点は少し減点。
                    score -= 0.05 * max(distance - 1.0, 0.0)

                    if score > best_score:
                        best_score = score
                        best = (y, x)

            if best is not None:
                additions.append(best)

        if not additions:
            break

        changed = False
        for y, x in additions:
            if accepted[y, x] == 0:
                accepted[y, x] = 255
                growth_added[y, x] = 255
                changed = True

        if not changed:
            break

    # 強いseedと元のbaseは絶対に保持。
    accepted[strong_bool] = 255
    accepted[base_bool] = 255

    result = mask_image(
        accepted,
        mask,
    )

    growth_added = mask_image(
        growth_added,
        mask,
    )

    return result, growth_added


# ============================================================
# 小成分・塊状成分除去
# ============================================================

def remove_small_components(
    binary,
    frangi_response,
    mask,
):
    """
    面積、骨格長、平均Frangi、細長さ、充填率を使用して
    小さなノイズ成分や塊状成分を除去する。

    threshold段階でseedを厳しくしたため、
    ここでは薄い血管の短い枝を消しすぎないよう、
    従来より少し緩めた設定を使用する。
    """

    binary = mask_image(
        binary,
        mask,
    )

    (
        num_labels,
        labels,
        stats,
        _,
    ) = cv2.connectedComponentsWithStats(
        binary,
        connectivity=8,
    )

    cleaned = np.zeros_like(
        binary,
        dtype=np.uint8,
    )

    for label_id in range(
        1,
        num_labels,
    ):
        component = (
            labels == label_id
        )

        area = int(
            stats[
                label_id,
                cv2.CC_STAT_AREA,
            ]
        )

        width = int(
            stats[
                label_id,
                cv2.CC_STAT_WIDTH,
            ]
        )

        height = int(
            stats[
                label_id,
                cv2.CC_STAT_HEIGHT,
            ]
        )

        if area < MIN_COMPONENT_AREA:
            continue

        component_skeleton = skeletonize(
            component
        )

        skeleton_length = int(
            np.count_nonzero(
                component_skeleton
            )
        )

        if skeleton_length < MIN_COMPONENT_LENGTH:
            continue

        mean_frangi = float(
            np.mean(
                frangi_response[
                    component
                ]
            )
        )

        if (
            mean_frangi
            < MIN_COMPONENT_MEAN_CONFIDENCE
        ):
            continue

        bounding_box_area = max(
            width * height,
            1,
        )

        fill_ratio = (
            area
            / bounding_box_area
        )

        elongation = (
            skeleton_length
            / max(
                np.sqrt(float(area)),
                1.0,
            )
        )

        # 短く、細長くなく、矩形内の充填率が高い成分は
        # 血管よりも塊状ノイズの可能性が高い。
        if (
            elongation < MIN_COMPONENT_ELONGATION
            and fill_ratio > MAX_COMPACT_FILL_RATIO
        ):
            continue

        cleaned[
            component
        ] = 255

    return mask_image(
        cleaned,
        mask,
    )


# ============================================================
# 局所高密度ノイズ除去
# ============================================================

def suppress_dense_noise_regions(
    binary,
    mask,
    window_size=41,
    max_local_density=0.35,
):
    """
    局所的に異常高密度な候補を除去する。

    視神経乳頭周辺や分岐領域も高密度になり得るため、
    初期設定では無効としている。
    """

    if window_size <= 1:
        return mask_image(
            binary,
            mask,
        )

    if window_size % 2 == 0:
        window_size += 1

    binary_float = (
        binary > 0
    ).astype(np.float32)

    mask_float = (
        mask > 0
    ).astype(np.float32)

    kernel = np.ones(
        (
            window_size,
            window_size,
        ),
        dtype=np.float32,
    )

    local_vessel = cv2.filter2D(
        binary_float,
        -1,
        kernel,
        borderType=cv2.BORDER_CONSTANT,
    )

    local_mask = cv2.filter2D(
        mask_float,
        -1,
        kernel,
        borderType=cv2.BORDER_CONSTANT,
    )

    local_density = (
        local_vessel
        / np.maximum(
            local_mask,
            1.0,
        )
    )

    dense_noise = (
        local_density
        > max_local_density
    )

    dense_noise_image = (
        dense_noise.astype(np.uint8)
        * 255
    )

    dilation_kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (5, 5),
    )

    dense_noise_image = cv2.dilate(
        dense_noise_image,
        dilation_kernel,
        iterations=1,
    )

    cleaned = binary.copy()

    cleaned[
        dense_noise_image > 0
    ] = 0

    return mask_image(
        cleaned,
        mask,
    )


# ============================================================
# 中心線化
# ============================================================

def make_centerline(
    vessel_binary,
    mask,
):
    """
    血管候補を軽く接続した後、1ピクセル幅に中心線化する。
    """

    vessel_binary = mask_image(
        vessel_binary,
        mask,
    )

    if CENTERLINE_CLOSE_SIZE > 1:
        kernel_size = CENTERLINE_CLOSE_SIZE

        if kernel_size % 2 == 0:
            kernel_size += 1

        kernel = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (
                kernel_size,
                kernel_size,
            ),
        )

        vessel_binary = safe_close(
            vessel_binary,
            kernel,
        )

        vessel_binary = mask_image(
            vessel_binary,
            mask,
        )

    skeleton = skeletonize(
        vessel_binary > 0
    )

    skeleton_image = (
        skeleton.astype(np.uint8)
        * 255
    )

    return mask_image(
        skeleton_image,
        mask,
    )


# ============================================================
# フラクタル次元
# ============================================================

def fractal_dimension(binary):
    """
    Box-counting法によるフラクタル次元。
    """

    binary_bool = binary > 0

    if np.count_nonzero(binary_bool) < 10:
        return 0.0

    minimum_side = min(
        binary_bool.shape
    )

    if minimum_side < 4:
        return 0.0

    power_size = (
        2
        ** int(
            np.floor(
                np.log2(
                    minimum_side
                )
            )
        )
    )

    binary_bool = binary_bool[
        :power_size,
        :power_size,
    ]

    sizes = (
        2
        ** np.arange(
            int(
                np.log2(
                    power_size
                )
            ),
            1,
            -1,
        )
    )

    counts = []

    for box_size in sizes:
        count = 0

        for y in range(
            0,
            power_size,
            box_size,
        ):
            for x in range(
                0,
                power_size,
                box_size,
            ):
                if np.any(
                    binary_bool[
                        y:y + box_size,
                        x:x + box_size,
                    ]
                ):
                    count += 1

        counts.append(count)

    counts = np.asarray(
        counts,
        dtype=np.float64,
    )

    sizes = np.asarray(
        sizes,
        dtype=np.float64,
    )

    valid = counts > 0

    if np.count_nonzero(valid) < 2:
        return 0.0

    coefficients = np.polyfit(
        np.log(
            sizes[valid]
        ),
        np.log(
            counts[valid]
        ),
        1,
    )

    return float(
        -coefficients[0]
    )


# ============================================================
# Overlay
# ============================================================

def create_overlay(
    original,
    skeleton,
    fundus_mask,
    line_thickness=OVERLAY_LINE_THICKNESS,
):
    if original.ndim == 2:
        overlay = cv2.cvtColor(
            original,
            cv2.COLOR_GRAY2BGR,
        )
    else:
        overlay = original.copy()

    skeleton = mask_image(
        skeleton,
        fundus_mask,
    )

    # 表示専用にskeletonを太くする。
    # 実際のvessel mask / skeletonデータは変更しない。
    line_thickness = max(
        1,
        int(line_thickness),
    )

    if line_thickness > 1:
        kernel = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (line_thickness, line_thickness),
        )

        skeleton = cv2.dilate(
            skeleton,
            kernel,
            iterations=1,
        )

        skeleton = mask_image(
            skeleton,
            fundus_mask,
        )

    skeleton_bool = (
        skeleton > 0
    )

    red = np.zeros_like(
        overlay,
        dtype=np.uint8,
    )

    # OpenCVはBGR。
    red[:, :, 2] = 255

    overlay_float = overlay.astype(
        np.float32
    )

    red_float = red.astype(
        np.float32
    )

    overlay_float[
        skeleton_bool
    ] = (
        overlay_float[
            skeleton_bool
        ]
        * (
            1.0
            - OVERLAY_ALPHA
        )
        + red_float[
            skeleton_bool
        ]
        * OVERLAY_ALPHA
    )

    return np.clip(
        overlay_float,
        0,
        255,
    ).astype(np.uint8)


# ============================================================
# 1画像処理
# ============================================================

def process_one_image(
    green_original,
    fundus_mask_original,
):
    original_shape = (
        green_original.shape
    )

    fundus_mask_original = np.where(
        fundus_mask_original > 0,
        255,
        0,
    ).astype(np.uint8)

    fundus_mask_original = (
        keep_largest_mask_component(
            fundus_mask_original
        )
    )

    # --------------------------------------------------------
    # 処理解像度へ変換
    # --------------------------------------------------------

    if PROCESS_SCALE < 0.999:
        new_width = max(
            1,
            int(
                round(
                    green_original.shape[1]
                    * PROCESS_SCALE
                )
            ),
        )

        new_height = max(
            1,
            int(
                round(
                    green_original.shape[0]
                    * PROCESS_SCALE
                )
            ),
        )

        green = cv2.resize(
            green_original,
            (
                new_width,
                new_height,
            ),
            interpolation=cv2.INTER_AREA,
        )

        fundus_mask = cv2.resize(
            fundus_mask_original,
            (
                new_width,
                new_height,
            ),
            interpolation=cv2.INTER_NEAREST,
        )

    else:
        green = green_original.copy()
        fundus_mask = fundus_mask_original.copy()

    fundus_mask = np.where(
        fundus_mask > 0,
        255,
        0,
    ).astype(np.uint8)

    effective_mask = create_effective_mask(
        fundus_mask
    )

    green = mask_image(
        green,
        effective_mask,
    )

    # --------------------------------------------------------
    # Green画像ノイズ推定
    # --------------------------------------------------------

    condition_info = estimate_image_condition(
        green,
        effective_mask,
    )

    green_noise = condition_info["noise_std"]
    image_condition = condition_info["condition"]

    # --------------------------------------------------------
    # 血管抽出用入力
    # --------------------------------------------------------
    # 前処理済みGreen画像を使う場合、ここで再び
    # bilateral / Gaussian / radial correctionをかけない。
    # 前処理側ですでに画像状態に応じたdenoiseと
    # illumination correctionを済ませているためである。
    if USE_PREPROCESSED_DIRECT:
        green_detection = mask_image(
            green,
            effective_mask,
        )
        green_radial_corrected = green_detection.copy()
        radial_banding = np.zeros_like(
            green_detection,
            dtype=np.uint8,
        )
    else:
        green_smooth = adaptive_smooth_green(
            green,
            effective_mask,
            image_condition,
        )

        # --------------------------------------------------------
        # 暗い画像の同心円バンディング除去
        # --------------------------------------------------------
        green_radial_corrected, radial_banding = remove_radial_banding(
            green_smooth,
            effective_mask,
            image_condition,
        )

    # --------------------------------------------------------
    # Frangi
    # --------------------------------------------------------

    (
        frangi_response,
        green_normalized,
    ) = build_frangi_response(
        green_radial_corrected,
        effective_mask,
        condition=image_condition,
        use_preprocessed_direct=USE_PREPROCESSED_DIRECT,
    )

    # --------------------------------------------------------
    # 局所コントラスト正規化
    # --------------------------------------------------------

    local_relative = build_local_relative_frangi(
        frangi_response,
        effective_mask,
    )

    # --------------------------------------------------------
    # Green画像を8x8に分割した局所平均による暗線検出
    # --------------------------------------------------------

    (
        local_darkness,
        local_darkness_z,
        local_dark_binary,
    ) = build_local_block_darkness(
        green_radial_corrected,
        effective_mask,
    )

    # 暗い画像で目立つ同心円状のアーティファクトを推定。
    # thresholdでは弱い候補だけを抑え、強い血管は残す。
    ring_score = build_ring_artifact_score(
        green_normalized,
        effective_mask,
    )

    # --------------------------------------------------------
    # Black-hat
    # --------------------------------------------------------

    blackhat = (
        build_multiscale_blackhat_response(
            green_normalized,
            effective_mask,
        )
    )

    # --------------------------------------------------------
    # 局所暗線
    # --------------------------------------------------------

    dark = build_dark_line_response(
        green_normalized,
        effective_mask,
    )

    # --------------------------------------------------------
    # 方向一貫性
    # --------------------------------------------------------

    coherence = build_line_coherence(
        green_normalized,
        effective_mask,
        condition=image_condition,
    )

    # --------------------------------------------------------
    # Frangi中心の候補抽出
    # --------------------------------------------------------

    (
        vessel_raw,
        high_threshold,
        low_threshold,
        noise_ratio,
        strong_seed,
        candidate,
    ) = threshold_vessels(
        frangi_response,
        blackhat,
        dark,
        coherence,
        local_relative,
        local_darkness_z,
        local_dark_binary,
        effective_mask,
        condition=image_condition,
        ring_score=ring_score,
    )

    # --------------------------------------------------------
    # Strong seedからの薄い血管成長
    # --------------------------------------------------------
    # ここで初めて、全体閾値より弱い応答を許可する。
    # strong seedから連続して到達できる場所だけを追加するため、
    # 暗い画像の独立した模様を一律に増やさない。
    vessel_grown, seed_growth_added = seed_guided_vessel_growth(
        strong_seed,
        vessel_raw,
        candidate,
        frangi_response,
        blackhat,
        dark,
        coherence,
        local_relative,
        ring_score,
        effective_mask,
        condition=image_condition,
    )

    # --------------------------------------------------------
    # 小成分・塊状成分除去
    # --------------------------------------------------------

    vessel_clean = remove_small_components(
        vessel_grown,
        frangi_response,
        effective_mask,
    )

    # --------------------------------------------------------
    # オプションの高密度ノイズ除去
    # --------------------------------------------------------

    density_limit = (
        DENSE_NOISE_MAX_DENSITY
    )

    if ENABLE_DENSE_NOISE_SUPPRESSION:
        if noise_ratio > FRANGI_NOISE_RATIO_LIMIT:
            density_limit = max(
                DENSE_NOISE_MAX_DENSITY_STRICT,
                DENSE_NOISE_MAX_DENSITY
                - (
                    noise_ratio
                    - FRANGI_NOISE_RATIO_LIMIT
                )
                * 0.5,
            )

        vessel_clean = (
            suppress_dense_noise_regions(
                vessel_clean,
                effective_mask,
                window_size=(
                    DENSE_NOISE_WINDOW_SIZE
                ),
                max_local_density=(
                    density_limit
                ),
            )
        )

        vessel_clean = remove_small_components(
            vessel_clean,
            frangi_response,
            effective_mask,
        )

    # --------------------------------------------------------
    # 処理解像度での中心線
    # --------------------------------------------------------

    skeleton = make_centerline(
        vessel_clean,
        effective_mask,
    )

    # --------------------------------------------------------
    # 元解像度へ戻す
    # --------------------------------------------------------

    if PROCESS_SCALE < 0.999:
        vessel_original = cv2.resize(
            vessel_clean,
            (
                original_shape[1],
                original_shape[0],
            ),
            interpolation=cv2.INTER_NEAREST,
        )

        effective_mask_original = cv2.resize(
            effective_mask,
            (
                original_shape[1],
                original_shape[0],
            ),
            interpolation=cv2.INTER_NEAREST,
        )

        effective_mask_original = np.where(
            effective_mask_original > 0,
            255,
            0,
        ).astype(np.uint8)

        vessel_original = mask_image(
            vessel_original,
            effective_mask_original,
        )

        skeleton_original = make_centerline(
            vessel_original,
            effective_mask_original,
        )

    else:
        vessel_original = vessel_clean.copy()
        skeleton_original = skeleton.copy()
        effective_mask_original = effective_mask.copy()

    # --------------------------------------------------------
    # 最終マスク再適用
    # --------------------------------------------------------

    final_mask = keep_largest_mask_component(
        fundus_mask_original
    )

    vessel_original = mask_image(
        vessel_original,
        final_mask,
    )

    skeleton_original = mask_image(
        skeleton_original,
        final_mask,
    )

    effective_mask_original = mask_image(
        effective_mask_original,
        final_mask,
    )

    return {
        "vessel_mask": vessel_original,
        "skeleton": skeleton_original,
        "frangi": frangi_response,
        "blackhat": blackhat,
        "dark": dark,
        "coherence": coherence,
        "local_relative": local_relative,
        "local_darkness": local_darkness,
        "local_darkness_z": local_darkness_z,
        "local_dark_binary": local_dark_binary,
        "ring_score": ring_score,
        "radial_banding": radial_banding,
        "raw": vessel_raw,
        "strong_seed": strong_seed,
        "candidate": candidate,
        "seed_growth": vessel_grown,
        "seed_growth_added": seed_growth_added,
        "effective_mask": effective_mask,
        "green_detection": green_radial_corrected,
        "effective_mask_original": effective_mask_original,
        "high_threshold": high_threshold,
        "low_threshold": low_threshold,
        "noise_ratio": noise_ratio,
        "green_noise": green_noise,
        "image_condition": image_condition,
        "image_mean": condition_info["mean"],
        "image_median": condition_info["median"],
        "image_contrast": condition_info["contrast"],
        "green_noise_ratio": condition_info["noise_ratio"],
        "density_limit": density_limit,
        "fractal_dimension": fractal_dimension(
            skeleton_original
        ),
    }


# ============================================================
# Debug画像
# ============================================================

# モンタージュ内の1タイルのサイズ(幅, 高さ)。
DEBUG_TILE_SIZE = (260, 260)

# モンタージュの列数。
DEBUG_MONTAGE_COLUMNS = 4

# タイルタイトルの見た目。
DEBUG_TITLE_FONT_SCALE = 0.55
DEBUG_TITLE_COLOR = (0, 255, 255)


def _prepare_debug_tile(
    title,
    image,
    tile_size=DEBUG_TILE_SIZE,
):
    """
    1枚のデバッグ画像を、タイトル付きの
    固定サイズBGRタイルに変換する。
    """

    if image.ndim == 2:
        visual = cv2.cvtColor(
            image,
            cv2.COLOR_GRAY2BGR,
        )
    else:
        visual = image.copy()

    visual = cv2.resize(
        visual,
        tile_size,
        interpolation=cv2.INTER_AREA,
    )

    # タイトルを読みやすくするため、
    # 背景に軽く帯を敷いてから文字を描く。
    cv2.rectangle(
        visual,
        (0, 0),
        (tile_size[0], 22),
        (0, 0, 0),
        thickness=-1,
    )

    cv2.putText(
        visual,
        title,
        (4, 16),
        cv2.FONT_HERSHEY_SIMPLEX,
        DEBUG_TITLE_FONT_SCALE,
        DEBUG_TITLE_COLOR,
        1,
        cv2.LINE_AA,
    )

    return visual


def build_debug_montage(
    images_with_titles,
    tile_size=DEBUG_TILE_SIZE,
    columns=DEBUG_MONTAGE_COLUMNS,
):
    """
    複数のデバッグ画像を1枚のグリッド画像にまとめる。

    段階ごとにファイルを分けず、1画像あたり1枚の
    montageファイルだけを見れば全段階を確認できるようにする。
    """

    tiles = [
        _prepare_debug_tile(title, image, tile_size)
        for title, image in images_with_titles
    ]

    if not tiles:
        return np.zeros(
            (tile_size[1], tile_size[0], 3),
            dtype=np.uint8,
        )

    rows = int(
        np.ceil(len(tiles) / columns)
    )

    grid_height = rows * tile_size[1]
    grid_width = columns * tile_size[0]

    grid = np.zeros(
        (grid_height, grid_width, 3),
        dtype=np.uint8,
    )

    for index, tile in enumerate(tiles):
        row = index // columns
        column = index % columns

        y0 = row * tile_size[1]
        y1 = y0 + tile_size[1]
        x0 = column * tile_size[0]
        x1 = x0 + tile_size[0]

        grid[y0:y1, x0:x1] = tile

    return grid


def save_debug_images(
    output_path,
    original,
    fundus_mask,
    result,
):
    """
    1画像分のデバッグ情報を1枚のmontage画像として保存する。

    strong seedとcandidateを別表示することで、
    「ノイズがseedになっているのか」
    「seedではないがcandidateに入っているのか」
    を確認できるようにする。
    """

    local_relative_vis = np.clip(
        result["local_relative"]
        * (
            255.0
            / max(
                LOCAL_CONTRAST_RATIO_THRESHOLD * 2.0,
                1e-3,
            )
        ),
        0,
        255,
    ).astype(np.uint8)

    overlay = create_overlay(
        original,
        result["skeleton"],
        fundus_mask,
        line_thickness=DEBUG_OVERLAY_LINE_THICKNESS,
    )

    local_dark_vis = np.clip(
        result["local_darkness_z"]
        * 80.0,
        0,
        255,
    ).astype(np.uint8)

    images_with_titles = [
        ("01 fundus_mask", fundus_mask),
        ("02 effective_mask", result["effective_mask"]),
        ("02b detection_input", result["green_detection"]),
        ("02c radial_banding", result["radial_banding"]),
        ("03 frangi", result["frangi"]),
        ("03b local_relative", local_relative_vis),
        ("03c local_darkness", local_dark_vis),
        ("03d local_dark_binary", result["local_dark_binary"]),
        ("04 blackhat", result["blackhat"]),
        ("05 dark_line", result["dark"]),
        ("06 coherence", result["coherence"]),
        ("06b ring_penalty", np.clip(result["ring_score"] * 255.0, 0, 255).astype(np.uint8)),
        ("07 strong_seed", result["strong_seed"]),
        ("08 candidate", result["candidate"]),
        ("08b seed_growth", result["seed_growth"]),
        ("08c growth_added", result["seed_growth_added"]),
        ("09 binary_raw", result["raw"]),
        ("10 vessel_mask", result["vessel_mask"]),
        ("11 skeleton", result["skeleton"]),
        ("12 overlay", overlay),
    ]

    montage = build_debug_montage(
        images_with_titles
    )

    save_image(
        output_path,
        montage,
    )


# ============================================================
# 一時出力から正式出力への移動
# ============================================================

def commit_outputs():
    clear_directory(
        OUTPUT_DIR
    )

    clear_directory(
        DEBUG_DIR
    )

    os.makedirs(
        TEMP_OUTPUT_DIR,
        exist_ok=True,
    )

    os.makedirs(
        TEMP_DEBUG_DIR,
        exist_ok=True,
    )

    for name in os.listdir(
        TEMP_OUTPUT_DIR
    ):
        shutil.move(
            os.path.join(
                TEMP_OUTPUT_DIR,
                name,
            ),
            os.path.join(
                OUTPUT_DIR,
                name,
            ),
        )

    for name in os.listdir(
        TEMP_DEBUG_DIR
    ):
        shutil.move(
            os.path.join(
                TEMP_DEBUG_DIR,
                name,
            ),
            os.path.join(
                DEBUG_DIR,
                name,
            ),
        )

    clear_directory(
        TEMP_OUTPUT_DIR
    )

    clear_directory(
        TEMP_DEBUG_DIR
    )


# ============================================================
# 一時データ削除
# ============================================================

def cleanup_temporary_outputs():
    clear_directory(
        TEMP_OUTPUT_DIR
    )

    clear_directory(
        TEMP_DEBUG_DIR
    )

    if os.path.exists(TEMP_CSV):
        os.remove(TEMP_CSV)


# ============================================================
# main
# ============================================================

def main():
    # --------------------------------------------------------
    # 入力フォルダ確認
    # --------------------------------------------------------

    if not os.path.isdir(INPUT_GREEN_DIR):
        raise FileNotFoundError(
            f"Green画像フォルダがありません: "
            f"{INPUT_GREEN_DIR}"
        )

    if not os.path.isdir(INPUT_MASK_DIR):
        raise FileNotFoundError(
            f"Fundus maskフォルダがありません: "
            f"{INPUT_MASK_DIR}"
        )

    # --------------------------------------------------------
    # 出力初期化
    # --------------------------------------------------------

    clear_directory(
        OUTPUT_DIR
    )

    clear_directory(
        DEBUG_DIR
    )

    cleanup_temporary_outputs()

    if os.path.exists(OUTPUT_CSV):
        os.remove(OUTPUT_CSV)

    # --------------------------------------------------------
    # Green画像一覧
    # --------------------------------------------------------

    files = sorted(
        [
            filename
            for filename in os.listdir(
                INPUT_GREEN_DIR
            )
            if filename.lower().endswith(
                ".png"
            )
        ]
    )

    if not files:
        raise FileNotFoundError(
            f"{INPUT_GREEN_DIR}に"
            "PNG画像がありません。"
        )

    # --------------------------------------------------------
    # Debug対象
    #
    # 常に先頭ALWAYS_DEBUG_COUNT枚を含める。
    # DEBUG_SAMPLE_COUNTがそれより多い場合のみ、
    # 残りをランダムに追加する。
    # --------------------------------------------------------

    # デバッグ対象は最初の5枚に固定。
    debug_files = set(
        files[:min(ALWAYS_DEBUG_COUNT, DEBUG_SAMPLE_COUNT)]
    )

    records = []
    skipped_count = 0

    try:
        # ====================================================
        # 全画像処理
        # ====================================================

        for filename in tqdm(
            files,
            desc="血管抽出",
        ):
            green_path = os.path.join(
                INPUT_GREEN_DIR,
                filename,
            )

            green = cv2.imread(
                green_path,
                cv2.IMREAD_GRAYSCALE,
            )

            if green is None:
                print(
                    "[SKIP] 読み込めませんでした: "
                    f"{green_path}"
                )

                skipped_count += 1
                continue

            base_id = get_base_image_id(
                filename
            )

            # ------------------------------------------------
            # Fundus mask
            # ------------------------------------------------

            (
                fundus_mask,
                mask_path,
            ) = load_saved_fundus_mask(
                base_id,
                green.shape,
            )

            if fundus_mask is None:
                print(
                    "[SKIP] 保存済みfundus maskがありません: "
                    f"{base_id}"
                )

                skipped_count += 1
                continue

            # ------------------------------------------------
            # 元画像
            # ------------------------------------------------

            original_path = find_original_image(
                base_id
            )

            if original_path is not None:
                original = cv2.imread(
                    original_path,
                    cv2.IMREAD_COLOR,
                )
            else:
                original = cv2.cvtColor(
                    green,
                    cv2.COLOR_GRAY2BGR,
                )

            if original is None:
                print(
                    "[SKIP] 元画像を読み込めません: "
                    f"{base_id}"
                )

                skipped_count += 1
                continue

            if original.shape[:2] != green.shape:
                original = cv2.resize(
                    original,
                    (
                        green.shape[1],
                        green.shape[0],
                    ),
                    interpolation=cv2.INTER_AREA,
                )

            # ------------------------------------------------
            # 血管抽出
            # ------------------------------------------------

            result = process_one_image(
                green,
                fundus_mask,
            )

            # ------------------------------------------------
            # 最終Overlay
            # ------------------------------------------------

            overlay = create_overlay(
                original,
                result["skeleton"],
                fundus_mask,
                line_thickness=OVERLAY_LINE_THICKNESS,
            )

            temporary_output_path = os.path.join(
                TEMP_OUTPUT_DIR,
                base_id + "_vessels.png",
            )

            temporary_mask_path = os.path.join(
                TEMP_OUTPUT_DIR,
                base_id + "_vessel_mask.png",
            )

            temporary_skeleton_path = os.path.join(
                TEMP_OUTPUT_DIR,
                base_id + "_skeleton.png",
            )

            save_image(
                temporary_output_path,
                overlay,
            )

            save_image(
                temporary_mask_path,
                result["vessel_mask"],
            )

            save_image(
                temporary_skeleton_path,
                result["skeleton"],
            )

            # ------------------------------------------------
            # Debug
            # ------------------------------------------------

            if filename in debug_files:
                montage_path = os.path.join(
                    TEMP_DEBUG_DIR,
                    base_id + "_debug.png",
                )

                save_debug_images(
                    montage_path,
                    original,
                    fundus_mask,
                    result,
                )

            # ------------------------------------------------
            # Metrics
            # ------------------------------------------------

            fundus_area = max(
                1,
                int(
                    np.count_nonzero(
                        fundus_mask
                    )
                ),
            )

            effective_area = max(
                1,
                int(
                    np.count_nonzero(
                        result[
                            "effective_mask_original"
                        ]
                    )
                ),
            )

            vessel_area = int(
                np.count_nonzero(
                    result["vessel_mask"]
                )
            )

            skeleton_length = int(
                np.count_nonzero(
                    result["skeleton"]
                )
            )

            records.append(
                {
                    "image": filename,
                    "base_id": base_id,
                    "fundus_mask": mask_path,
                    "original_image": (
                        original_path
                        if original_path is not None
                        else ""
                    ),
                    "output_image": os.path.join(
                        OUTPUT_DIR,
                        base_id
                        + "_vessels.png",
                    ),
                    "fundus_pixels": fundus_area,
                    "effective_mask_pixels": (
                        effective_area
                    ),
                    "vessel_area_ratio": (
                        vessel_area
                        / fundus_area
                    ),
                    "vessel_area_ratio_effective": (
                        vessel_area
                        / effective_area
                    ),
                    "skeleton_length_ratio": (
                        skeleton_length
                        / fundus_area
                    ),
                    "skeleton_length_ratio_effective": (
                        skeleton_length
                        / effective_area
                    ),
                    "vessel_pixels": vessel_area,
                    "skeleton_pixels": skeleton_length,
                    "fractal_dimension": (
                        result[
                            "fractal_dimension"
                        ]
                    ),
                    "frangi_high_percentile": (
                        FRANGI_HIGH_PERCENTILE
                    ),
                    "frangi_low_percentile": (
                        FRANGI_LOW_PERCENTILE
                    ),
                    "actual_high_threshold": (
                        result[
                            "high_threshold"
                        ]
                    ),
                    "actual_low_threshold": (
                        result[
                            "low_threshold"
                        ]
                    ),
                    "noise_ratio": (
                        result[
                            "noise_ratio"
                        ]
                    ),
                    "green_noise": (
                        result[
                            "green_noise"
                        ]
                    ),
                    "image_condition": (
                        result[
                            "image_condition"
                        ]
                    ),
                    "image_mean": (
                        result[
                            "image_mean"
                        ]
                    ),
                    "image_median": (
                        result[
                            "image_median"
                        ]
                    ),
                    "image_contrast": (
                        result[
                            "image_contrast"
                        ]
                    ),
                    "green_noise_ratio": (
                        result[
                            "green_noise_ratio"
                        ]
                    ),
                    "seed_growth_pixels": int(
                        np.count_nonzero(
                            result["seed_growth_added"]
                        )
                    ),
                    "seed_growth_enabled": (
                        SEED_GROWTH_ENABLED
                    ),
                    "ring_penalty_mean": float(
                        np.mean(
                            result["ring_score"][
                                result["effective_mask_original"] > 0
                            ]
                        )
                    ),
                    "coherence_min": (
                        COHERENCE_MIN
                    ),
                    "local_contrast_ratio_threshold": (
                        LOCAL_CONTRAST_RATIO_THRESHOLD
                    ),
                    "local_dark_grid_size": (
                        LOCAL_DARK_GRID_SIZE
                    ),
                    "local_dark_min_drop": (
                        LOCAL_DARK_MIN_DROP
                    ),
                    "local_dark_min_z": (
                        LOCAL_DARK_MIN_Z
                    ),
                    "local_dark_pixels": int(
                        np.count_nonzero(
                            result["local_dark_binary"]
                        )
                    ),
                    "dense_noise_enabled": (
                        ENABLE_DENSE_NOISE_SUPPRESSION
                    ),
                    "density_limit": (
                        result[
                            "density_limit"
                        ]
                    ),
                    "process_scale": (
                        PROCESS_SCALE
                    ),
                    "mask_touches_border": (
                        mask_touches_border(
                            fundus_mask
                        )
                    ),
                }
            )

        # ====================================================
        # 処理結果確認
        # ====================================================

        if not records:
            raise RuntimeError(
                "処理できた画像がありません。"
                "preprocessed_green_imagesと"
                "fundus_masksを確認してください。"
            )

        # ====================================================
        # CSV
        # ====================================================

        dataframe = pd.DataFrame(
            records
        )

        dataframe = dataframe.sort_values(
            "fractal_dimension",
            ascending=False,
        ).reset_index(
            drop=True
        )

        dataframe["rank"] = np.arange(
            1,
            len(dataframe) + 1,
        )

        dataframe.to_csv(
            TEMP_CSV,
            index=False,
            encoding="utf-8-sig",
        )

        # ====================================================
        # 全処理終了後に正式出力へ移動
        # ====================================================

        commit_outputs()

        shutil.move(
            TEMP_CSV,
            OUTPUT_CSV,
        )

    except KeyboardInterrupt:
        cleanup_temporary_outputs()

        print()
        print("処理が中断されました。")
        print(
            "今回の未完成データは"
            "正式出力へ保存していません。"
        )

        raise

    except Exception:
        cleanup_temporary_outputs()
        raise

    # ========================================================
    # 終了表示
    # ========================================================

    print()
    print(
        "========================================"
    )
    print(
        "血管抽出が完了しました"
    )
    print(
        "========================================"
    )

    print(
        f"入力Green:          "
        f"{INPUT_GREEN_DIR}"
    )

    print(
        f"入力fundus mask:    "
        f"{INPUT_MASK_DIR}"
    )

    print(
        f"元画像:             "
        f"{ORIGINAL_IMAGE_DIR}"
    )

    print(
        f"最終出力:           "
        f"{OUTPUT_DIR}"
    )

    print(
        f"デバッグ:           "
        f"{DEBUG_DIR}"
    )

    print(
        f"CSV:                "
        f"{OUTPUT_CSV}"
    )

    print(
        f"処理画像数:          "
        f"{len(dataframe)}"
    )

    print(
        f"スキップ数:          "
        f"{skipped_count}"
    )

    print(
        f"PROCESS_SCALE:       "
        f"{PROCESS_SCALE}"
    )

    print(
        f"Preprocessed direct:  "
        f"{USE_PREPROCESSED_DIRECT}"
    )

    print(
        f"Coherence sigma:      "
        f"normal={COHERENCE_SIGMA_NORMAL}, dark={COHERENCE_SIGMA_DARK}"
    )

    print(
        f"Low percentile:      "
        f"{FRANGI_LOW_PERCENTILE}"
    )

    print(
        f"Coherence minimum:   "
        f"{COHERENCE_MIN}"
    )

    print(
        f"Local contrast ratio:"
        f"{LOCAL_CONTRAST_RATIO_THRESHOLD}"
    )

    print(
        f"Local dark grid:      "
        f"{LOCAL_DARK_GRID_SIZE}x{LOCAL_DARK_GRID_SIZE}"
    )

    print(
        f"Local dark min drop:  "
        f"{LOCAL_DARK_MIN_DROP}"
    )

    print(
        f"Overlay thickness:    "
        f"{OVERLAY_LINE_THICKNESS}"
    )

    print(
        f"Debug overlay thick:  "
        f"{DEBUG_OVERLAY_LINE_THICKNESS}"
    )

    print(
        f"Dense suppression:   "
        f"{ENABLE_DENSE_NOISE_SUPPRESSION}"
    )

    print()
    print(
        dataframe.head(10).to_string(
            index=False
        )
    )


# ============================================================
# 実行
# ============================================================

if __name__ == "__main__":
    main()


血管抽出:   0%|          | 7/1434 [15:13<51:42:14, 130.44s/it]



処理が中断されました。
今回の未完成データは正式出力へ保存していません。


KeyboardInterrupt: 